# Relaxation of Explicit Water Systems
https://ambermd.org/tutorials/basic/tutorial13/index.php

Tutorial uses the human RAMP1 extracellular domain (PDB ID: 2YX8) monomer in an explicit solvent system containing the OPC water model\
but my system contains 2 proteins that are
1. interacting(?) - has interface residues
2. contained missing residues - extrapolated with `modelller`

NOTE: have to think of how to simulate with the stabilizer (FSC)...

NOTE: all bash commands run in the terminal of `/home/wjoon21/project/chen2023_possible_allosteric/B2/trial5_tleap_script_parm7_rst7/`

# 1. Minimize the added water and ions
Input files:
1. `1min.in`
2. `amber_o98_ion.parm7`
3. `amber_o98_ion.rst7`

Terminal command:
```sh
$AMBERHOME/bin/pmemd -O -i 1min.in -o 1min.out -p amber_o98_ion.parm7 -c amber_o98_ion.rst7 -r 1min.rst7 -inf 1min.info -ref amber_o98_ion.rst7 -x mdcrd.1min
```

## Trial(s) 1
10/06/2024

FAIL

`1min.in` from tutorial:
```in
minimization of solvent
 &cntrl
  imin = 1, maxcyc = 1000,   
  ncyc = 20, ntx = 1,                     
  ntwe = 0, ntwr = 500, ntpr = 50,
  ntc = 2, ntf = 2, ntb = 1, ntp = 0,
  cut = 10.0,   
  ntr=1, restraintmask = ':1-81',
  restraint_wt = 100.,
  ioutfm=1, ntxo=2,
 /

```
NOTE: LINEBREAK AFTER THE LAST `/` IS AN IMPORTANT SYNTAX

Key settings:
```
imin = 1       This run is a minimization run.
maxcyc = 1000  The maximum amount of minimization cycles is 1000.
ncyc = 20      The first 20 cycles will utilize the steepest descent 
                  algorithm before shifting to the conjugate gradient 
                  algorithm for the remaining cycles. 
ntx = 1        The coordinates but not velocities are read formatted 
                  from the coordinate file provided. 
ntwe = 0       No mden files are written.
ntwr = 500     The amount of steps in which "restrt" files are written.
ntpr = 50      The amount of steps in which "mdout" and "mdinfo" files
                  are written.
ntc = 2        Bonds involving H are constrained. 
ntf = 2        Bonds involving H are omitted from force evaluation. 
ntb = 1        There is constant volume.
ntp = 0        There is no pressure scaling.
cut = 10.0     The non-bonded cutoff is 10.0 Angstroms.
ntr = 1        Use restraints.
restraintmask = ':1-81'  Restrain the solute - protein (res 1-81).
restraint_wt = 100  Positional restraint is 100 kcal/mol*Ang^-2.
ioutfm = 1     The format of the coordinate and velocity trajectory 
                 files written as binary NetCDF.
ntxo=2         The "restrt" file format is NetCDF.
```

Revise the followings:
1. Change `restraintmask` (residues to be restrained) based on `amber_o98_wions_water.pdb` (294) $\rightarrow$ should be correct
2. Specify smaller simulation timestep `dx0` (default `None`)
3. Change to `ntpr=1` from `50`  $\rightarrow$ to see which exact step raises the error
```in
minimization of solvent
 &cntrl
  imin = 1, maxcyc = 1000,   
  ncyc = 20, ntx = 1,                     
  ntwe = 0, ntwr = 500, ntpr = 1,
  ntc = 2, ntf = 2, ntb = 1, ntp = 0,
  cut = 10.0,   
  ntr=0, 
  restraintmask = ':1-294',
  restraint_wt = 100.,
  ioutfm=1, ntxo=2,
  dx0=1.0D-10,
 /

```

FAIL: `STOP PMEMD Terminated Abnormally!`
```out
     Coordinate resetting cannot be accomplished,
     deviation is too large
     iter_cnt, my_bond_idx, i and j are :       3    1787    3586    3588
  *** Especially for minimization, try ntc=1 (no shake)
```
GLN residue (229) has steric clashes on gamma hydrogens?

Check in VMD $\rightarrow$ NO noticeable problems?

Remove restraints (`ntr=1, restraintmask = ':1-294', restraint_wt = 100.,`) and non-essential(?) parameters (`ioutfm=1, ntxo=2,`)

FAIL: 
```
Program received signal SIGSEGV: Segmentation fault - invalid memory reference.

Backtrace for this error:
#0  0x7f564e517d11 in ???
#1  0x7f564e516ee5 in ???
#2  0x7f564e32d08f in ???
        at /build/glibc-LcI20x/glibc-2.31/signal/../sysdeps/unix/sysv/linux/x86_64/sigaction.c:0
#3  0x55b4eb82df9a in ???
#4  0x55b4eb8232fd in ???
#5  0x55b4eb85a978 in ???
#6  0x55b4eb88be3b in ???
#7  0x55b4eb75d9ae in ???
#8  0x7f564e30e082 in __libc_start_main
        at ../csu/libc-start.c:308
#9  0x55b4eb75d9ed in ???
#10  0xffffffffffffffff in ???
Segmentation fault (core dumped)
``` 
on all kinds of input topology (parameter) and coordinate (restart) files

But the unaltered `1min.in` runs successfully for the tutorial system?

NOT a problem with AMBER version

## Trial 2
Revise `1min.in` according to Christian's suggestion

## Minimization input

Remove `ntc=2` (bonds involving H are constrained) and `ntf=2` (bonds involving H are omitted from force evaluation)\
i.e., set to default (1, SHAKE is not performed) $\because$ `modeller` may have introduced hydrogen bonds that are slightly "weird"

Before (only edited `restraintmask` residue numbers from the tutorial):
```in
minimization of solvent
 &cntrl
  imin = 1, maxcyc = 1000,   
  ncyc = 20, ntx = 1,                     
  ntwe = 0, ntwr = 500, ntpr = 50,
  ntc = 2, ntf = 2, ntb = 1, ntp = 0,
  cut = 10.0,   
  ntr=1, restraintmask = ':1-294',
  restraint_wt = 100.,
  ioutfm=1, ntxo=2,
 /

```
After:
```in
minimization of solvent
 &cntrl
  imin = 1, maxcyc = 1000,   
  ncyc = 20, ntx = 1,                     
  ntwe = 0, ntwr = 500, ntpr = 50,
  ntb = 1, ntp = 0,
  cut = 10.0,   
  ntr=1, restraintmask = ':1-294',
  restraint_wt = 100.,
  ioutfm=1, ntxo=2,
 /

```

## Minimization output

SUCCESS

`1min.out` (end)
```out
   NSTEP       ENERGY          RMS            GMAX         NAME    NUMBER
   1000      -2.3087E+05     5.6532E-01     7.2882E+01     CD       2390

 BOND    =    42092.9630  ANGLE   =      590.0988  DIHED      =     1149.6412
 UB      =        0.0000  IMP     =        0.0000  CMAP       =      742.1969
 VDWAALS =    47902.0356  EEL     =  -338499.6135  HBOND      =        0.0000
 1-4 VDW =     1580.1405  1-4 EEL =    12496.4453  RESTRAINT  =     1073.6456
 EAMBER  =  -231946.0924


  Maximum number of minimization cycles reached.


                    FINAL RESULTS



   NSTEP       ENERGY          RMS            GMAX         NAME    NUMBER
   1000      -2.3087E+05     5.6532E-01     7.2882E+01     CD       2390

 BOND    =    42092.9630  ANGLE   =      590.0988  DIHED      =     1149.6412
 UB      =        0.0000  IMP     =        0.0000  CMAP       =      742.1969
 VDWAALS =    47902.0356  EEL     =  -338499.6135  HBOND      =        0.0000
 1-4 VDW =     1580.1405  1-4 EEL =    12496.4453  RESTRAINT  =     1073.6456
 EAMBER  =  -231946.0924
--------------------------------------------------------------------------------
   5.  TIMINGS
--------------------------------------------------------------------------------

|  NonSetup CPU Time in Major Routines:
|
|     Routine           Sec        %
|     ------------------------------
|     Nonbond         260.41   98.56
|     Bond              0.44    0.17
|     Angle             0.36    0.14
|     Dihedral          2.14    0.81
|     Shake             0.00    0.00
|     Other             0.86    0.32
|     ------------------------------
|     Total           264.20

|  PME Nonbond Pairlist CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     Set Up Cit           0.04    0.02
|     Build List           8.06    3.05
|     ---------------------------------
|     Total                8.10    3.07

|  PME Direct Force CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     NonBonded Calc     218.74   82.79
|     Exclude Masked       2.05    0.77
|     Other                0.45    0.17
|     ---------------------------------
|     Total              221.24   83.74

|  PME Reciprocal Force CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     1D bspline           1.63    0.62
|     Grid Charges         2.39    0.90
|     Scalar Sum           9.25    3.50
|     Gradient Sum         4.56    1.72
|     FFT                 12.83    4.86
|     ---------------------------------
|     Total               30.65   11.60

|  Setup CPU time:            0.39 seconds
|  NonSetup CPU time:       264.20 seconds
|  Total CPU time:          264.59 seconds     0.07 hours

|  Setup wall time:           1    seconds
|  NonSetup wall time:      264    seconds
|  Total wall time:         265    seconds     0.07 hours
```
Output: `1min.rst7`

## Convert `parm7` and `rst7` files into a `pdb` file

`cpptraj_netCDF_to_pdb.in`
```in
parm amber_o98_ion.parm7
trajin 1min.rst7
trajout 1min_rst7.pdb PDB
```

In [ ]:
!$AMBERHOME/bin/cpptraj -i cpptraj_netCDF_to_pdb.in>cpptraj_netCDF_to_pdb.out &

Terminal:
```
[1] 78352
```
`cpptraj_netCDF_to_pdb.out`
```out
INPUT: Reading input from 'cpptraj_netCDF_to_pdb.in'
  [parm amber_o98_ion.parm7]
	Reading 'amber_o98_ion.parm7' as Amber Topology
	Radius Set: modified Bondi radii (mbondi)
  [trajin 1min.rst7]
	Reading '1min.rst7' as Amber NC Restart
TIME: Total execution time: 0.0686 seconds.
```
But no output `1min_rst7.pdb` file...?

Seems like script in tutorial is WRONG

Try the script and command in `02_.ipynb` (overwrite existing)

`cpptraj_netCDF_to_pdb.in`
```in
trajin 1min.rst7
trajout amber_o98_1min_rst7.pdb PDB
run
```

In [ ]:
!cpptraj -p amber_o98_ion.parm7 -i cpptraj_netCDF_to_pdb.in > cpptraj_netCDF_to_pdb.out

SUCCESS

`cpptraj_netCDF_to_pdb.out`
```out
	Reading 'amber_o98_ion.parm7' as Amber Topology
	Radius Set: modified Bondi radii (mbondi)
INPUT: Reading input from 'cpptraj_netCDF_to_pdb.in'
  [trajin 1min.rst7]
	Reading '1min.rst7' as Amber NC Restart
  [trajout amber_o98_1min_rst7.pdb PDB]
	Writing 'amber_o98_1min_rst7.pdb' as PDB
---------- RUN BEGIN -------------------------------------------------

PARAMETER FILES (1 total):
 0: amber_o98_ion.parm7, 56610 atoms, 13351 res, box: Truncated octahedron, 13059 mol, 12955 solvent

INPUT TRAJECTORIES (1 total):
 0: '1min.rst7' is a NetCDF AMBER restart file with coordinates, time, box, Parm amber_o98_ion.parm7 (Truncated octahedron box) (reading 1 of 1)
  Coordinate processing will occur on 1 frames.

OUTPUT TRAJECTORIES (1 total):
  'amber_o98_1min_rst7.pdb' (1 frames) is a PDB file

BEGIN TRAJECTORY PROCESSING:
Warning: No PDB space group specified.
Warning: Topology 'amber_o98_ion.parm7' has 12955 extra points
Warning:   that will not be included in output PDB.
Warning: To include them, specify 'include_ep'. Otherwise, use output PDB as
Warning:   topology or create a new topology with 'strip' or 'parmstrip'.
.....................................................
ACTIVE OUTPUT TRAJECTORIES (1):
  amber_o98_1min_rst7.pdb (coordinates, time, box)
----- 1min.rst7 (1-1, 1) -----
100% Complete.

Read 1 frames and processed 1 frames.
TIME: Avg. throughput= 11.0217 frames / second.

ACTION OUTPUT:
TIME: Analyses took 0.0000 seconds.

RUN TIMING:
TIME:		Init               : 0.0000 s (  0.01%)
TIME:		Trajectory Process : 0.0907 s ( 99.90%)
TIME:		Action Post        : 0.0000 s (  0.00%)
TIME:		Analysis           : 0.0000 s (  0.00%)
TIME:		Data File Write    : 0.0000 s (  0.00%)
TIME:		Other              : 0.0001 s (  0.00%)
TIME:	Run Total 0.0908 s
---------- RUN END ---------------------------------------------------
TIME: Total execution time: 0.1307 seconds.
--------------------------------------------------------------------------------
```
Output:

`amber_o98_1min_rst7.pdb`

NOTE: check on VMD (12/06) to see if the minimized solvent and protein looks OK

NOTE: change input and output file names to auto-sort by process number (`1min_`)

Protein + solvent:
```
Style        Color   Selection       Material
NewCartoon   Name    all not water   Opaque
QuickSurf    Name    water           Transparent
```
Exporting publication-quality figures:
http://www.ks.uiuc.edu/Training/Tutorials/vmd/tutorial-html/node2.html#SECTION00025000000000000000

White background:
```
Graphics -> Color -> Display -> Background -> 8 white
```
Saving `.png`:
```
File -> Render -> Snapshot (VMD OpenGL window) -> /path/to/<file_name>.png -> display %s -> Start Rendering
```
Output

![1min_rst_pdb_quicksurf](B2/trial5_tleap_script_parm7_rst7/1min_rst_pdb_quicksurf.png)

# 2. Heat up the system from 100 K under constant volume 

## Heating input

Input files:
1. `2mdheat.in`
2. `amber_o98_ion.parm7`
3. `1min.rst7`

Terminal command:
```sh
$AMBERHOME/bin/pmemd.cuda -O -i 2mdheat.in -o 2mdheat.out -p amber_o98_ion.parm7 -c 1min.rst7 -r 2mdheat.rst7 -inf 2mdheat.info -ref 1min.rst7 -x mdcrd.2mdheat
```

`2mdheat.in` from tutorial:
```in
 &cntrl
  imin = 0, nstlim = 1000000, dt = 0.001,
  irest = 0, ntx = 1, ig = -1,
  tempi = 100.0, temp0 = 298.0,
  ntc = 2, ntf = 2, tol = 0.00001,
  ntwx = 10000, ntwe = 0, ntwr = 1000, ntpr = 1000,
  cut = 8.0, iwrap = 0,
  ntt =3, gamma_ln=1., ntb = 1, ntp = 0,
  nscm = 0,
  ntr=1, restraintmask=':1-81', restraint_wt=100.0
  nmropt=1,
  ioutfm=1, ntxo=2,
 /
&wt TYPE="TEMP0", istep1=0, istep2=1000000, value1=100., value2=298., /
&wt TYPE="END", /

```
Key settings:
```
imin = 0          This is not a minimization run.
nstlim = 1000000  There will be 1000000 MD-steps. 
dt = 0.001        There is a time step of 1 femtosecond. 
irest = 0         The simulation will not be restarted.
ntx = 1           The coordinates but not velocities are read 
                    formatted from the coordinate file provided.
ig = -1           The random seed number is based on the current 
                    date and time of the run.  
tempi = 100.0     The initial temperature is 100.0 K.
temp0 = 298.0     The reference temperature is set to 298.0 K 
ntc = 2           Bonds involving H are constrained. 
ntf = 2           Bonds involving H are omitted from force evaluation. 
tol = 0.00001     The error of tolerance is 0.00001 Angstroms.
ntwx = 10000      The coordinates are written to a mdcrd file 10000 times. 
ntwe = 0          No mden files are written.
ntwr = 1000       Number of steps in which "restrt" files are written. 
ntpr = 1000       Number of steps in which "mdout" and "mdinfo" 
                    files are written.
cut = 8.0         The non-bonded cutoff is 8.0 Angstroms.
iwrap = 0         No wrapping is performed.
ntt = 3           Langevin thermostat for temperature control is set.
gamma_ln=1.       The collision frequency gamma is set to 1 picosecond. 
ntb = 1           There is constant volume.
ntp = 0           There is no pressure scaling.
nscm = 0          There is no removal of center of mass motion.
ntr=1             Atoms which are restrained within the simulation.
restraintmask = ':1-81' Restrain solute (as in 1min.in).
restraint_wt = 100  Positional restraint is 100 kcal/mol*Ang^-2.
nmropt=1          NMR restraints and weight changes will be read. 
ioutfm = 1        The format of the coordinate and velocity trajectory
                    files written as binary NetCDF.
ntxo=2            The "restrt" file format is NetCDF.

TYPE="TEMP0"                  The target T will vary.
istep1=0, istep2=1000000      Change in T will occur in 1000000 increments.
value1=100., value2=298.      T begins at 100.0 K and increases to 298.0 K.
```

Change `restraintmask` (residues to be restrained) based on `amber_o98_wions_water.pdb` (294):
```in
&cntrl
  imin = 0, nstlim = 1000000, dt = 0.001,
  irest = 0, ntx = 1, ig = -1,
  tempi = 100.0, temp0 = 298.0,
  ntc = 2, ntf = 2, tol = 0.00001,
  ntwx = 10000, ntwe = 0, ntwr = 1000, ntpr = 1000,
  cut = 8.0, iwrap = 0,
  ntt =3, gamma_ln=1., ntb = 1, ntp = 0,
  nscm = 0,
  ntr=1, restraintmask=':1-294', restraint_wt=100.0
  nmropt=1,
  ioutfm=1, ntxo=2,
 /
&wt TYPE="TEMP0", istep1=0, istep2=1000000, value1=100., value2=298., /
&wt TYPE="END", /

```

## Heating output

SUCCESS(?)

Terminal:
```sh
Note: The following floating-point exceptions are signalling: IEEE_UNDERFLOW_FLAG IEEE_DENORMAL
```
1. "You can generally ignore such messages." http://archive.ambermd.org/202108/0046.html
2. "this message appears during the simulation when you use a `pmemd.cuda` version compiled with newer versions of the Fortran compiler" https://github.com/GHeinzelmann/BAT.py/issues/22

`2heatmd.out`
```out
===============================================================================

      A V E R A G E S   O V E R    1000 S T E P S


 NSTEP =  1000000   TIME(PS) =    1000.000  TEMP(K) =   199.01  PRESS =     0.0
 Etot   =   -172995.5848  EKtot   =     17748.8550  EPtot      =   -190744.4398
 BOND   =       461.6960  ANGLE   =      1257.2685  DIHED      =      1280.0435
 UB     =         0.0000  IMP     =         0.0000  CMAP       =       744.1267
 1-4 NB =      1535.8202  1-4 EEL =     12312.0164  VDWAALS    =     28308.3453
 EELEC  =   -238249.7368  EHBOND  =         0.0000  RESTRAINT  =      1605.9804
 EAMBER (non-restraint)  =   -192350.4202
 ------------------------------------------------------------------------------

 NMR restraints: Bond =    0.000   Angle =     0.000   Torsion =     0.000
===============================================================================

      R M S  F L U C T U A T I O N S


 NSTEP =  1000000   TIME(PS) =    1000.000  TEMP(K) =    57.02  PRESS =     0.0
 Etot   =     11506.1918  EKtot   =      5085.1776  EPtot      =      6575.9087
 BOND   =       101.2503  ANGLE   =       166.5223  DIHED      =        28.9594
 UB     =         0.0000  IMP     =         0.0000  CMAP       =         4.4026
 1-4 NB =        12.5400  1-4 EEL =        14.7956  VDWAALS    =      2560.5463
 EELEC  =      8492.8824  EHBOND  =         0.0000  RESTRAINT  =       357.6888
 EAMBER (non-restraint)  =      6218.2198
 ------------------------------------------------------------------------------



 NMR restraints on final step:

--------------------------------------------------------------------------------
   5.  TIMINGS
--------------------------------------------------------------------------------

|  NonSetup CPU Time in Major Routines:
|
|     Routine           Sec        %
|     ------------------------------
|     Nonbond          82.26    4.22
|     Bond              0.00    0.00
|     Angle             0.00    0.00
|     Dihedral          0.00    0.00
|     Shake             2.68    0.14
|     RunMD          1864.39   95.55
|     Other             1.94    0.10
|     ------------------------------
|     Total          1951.27

|  PME Nonbond Pairlist CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     Set Up Cit           0.00    0.00
|     Build List           0.00    0.00
|     ---------------------------------
|     Total                0.00    0.00

|  PME Direct Force CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     NonBonded Calc       0.00    0.00
|     Exclude Masked       0.00    0.00
|     Other                1.27    0.07
|     ---------------------------------
|     Total                1.27    0.07

|  PME Reciprocal Force CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     1D bspline           0.00    0.00
|     Grid Charges         0.00    0.00
|     Scalar Sum           0.00    0.00
|     Gradient Sum         0.00    0.00
|     FFT                  0.00    0.00
|     ---------------------------------
|     Total                0.00    0.00

|  Final Performance Info:
|     -----------------------------------------------------
|     Average timings for last    9000 steps:
|     Elapsed(s) =      17.97 Per Step(ms) =       2.00
|         ns/day =      43.28   seconds/ns =    1996.41
|
|     Average timings for all steps:
|     Elapsed(s) =    1951.17 Per Step(ms) =       1.95
|         ns/day =      44.28   seconds/ns =    1951.17
|     -----------------------------------------------------

|  Setup CPU time:            0.59 seconds
|  NonSetup CPU time:      1951.27 seconds
|  Total CPU time:         1951.86 seconds     0.54 hours

|  Setup wall time:           1    seconds
|  NonSetup wall time:     1951    seconds
|  Total wall time:        1952    seconds     0.54 hours
```

## Convert `parm7` and `rst7` to `pdb` for visualization

`cpptraj_netCDF_to_pdb.in`
```in
trajin 2mdheat.rst7
trajout 2mdheat_rst7.pdb PDB
run
```

In [ ]:
!cpptraj -p amber_o98_ion.parm7 -i 2mdheat_cpptraj_netCDF_to_pdb.in > 2mdheat_cpptraj_netCDF_to_pdb.out

SUCCESS

`2mdheat_cpptraj_netCDF_to_pdb.out`
```out
	Reading 'amber_o98_ion.parm7' as Amber Topology
	Radius Set: modified Bondi radii (mbondi)
INPUT: Reading input from '2mdheat_cpptraj_netCDF_to_pdb.in'
  [trajin 2mdheat.rst7]
	Reading '2mdheat.rst7' as Amber NC Restart
  [trajout 2mdheat_rst7.pdb PDB]
	Writing '2mdheat_rst7.pdb' as PDB
---------- RUN BEGIN -------------------------------------------------

PARAMETER FILES (1 total):
 0: amber_o98_ion.parm7, 56610 atoms, 13351 res, box: Truncated octahedron, 13059 mol, 12955 solvent

INPUT TRAJECTORIES (1 total):
 0: '2mdheat.rst7' is a NetCDF AMBER restart file with coordinates, velocities, time, box, Parm amber_o98_ion.parm7 (Truncated octahedron box) (reading 1 of 1)
  Coordinate processing will occur on 1 frames.

OUTPUT TRAJECTORIES (1 total):
  '2mdheat_rst7.pdb' (1 frames) is a PDB file

BEGIN TRAJECTORY PROCESSING:
Warning: No PDB space group specified.
Warning: Topology 'amber_o98_ion.parm7' has 12955 extra points
Warning:   that will not be included in output PDB.
Warning: To include them, specify 'include_ep'. Otherwise, use output PDB as
Warning:   topology or create a new topology with 'strip' or 'parmstrip'.
.....................................................
ACTIVE OUTPUT TRAJECTORIES (1):
  2mdheat_rst7.pdb (coordinates, velocities, time, box)
----- 2mdheat.rst7 (1-1, 1) -----
100% Complete.

Read 1 frames and processed 1 frames.
TIME: Avg. throughput= 6.7165 frames / second.

ACTION OUTPUT:
TIME: Analyses took 0.0000 seconds.

RUN TIMING:
TIME:		Init               : 0.0000 s (  0.00%)
TIME:		Trajectory Process : 0.1489 s ( 99.91%)
TIME:		Action Post        : 0.0000 s (  0.00%)
TIME:		Analysis           : 0.0000 s (  0.00%)
TIME:		Data File Write    : 0.0000 s (  0.00%)
TIME:		Other              : 0.0001 s (  0.00%)
TIME:	Run Total 0.1490 s
---------- RUN END ---------------------------------------------------
TIME: Total execution time: 0.2094 seconds.
--------------------------------------------------------------------------------
```
http://archive.ambermd.org/202207/0053.html \
`Warning: No PDB space group specified.` can be safely ignored

Output:

`2mdheat_rst7.pdb` on VMD\
![2mdheat_rst_pdb_quicksurf](B2/trial5_tleap_script_parm7_rst7/2mdheat_rst_pdb_quicksurf.png)\
The water QuickSurf became disorganized(?) $\rightarrow$ Does heating cause this?

# 3. Relax the system at a constant pressure

## MD Input

Input files:
1. `3md.in`
2. `amber_o98_ion.parm7`
3. `2mdheat.rst7`

Terminal command:
```sh
$AMBERHOME/bin/pmemd.cuda -O -i 3md.in -o 3md.out -p amber_o98_ion.parm7 -c 2mdheat.rst7 -r 3md.rst7 -inf 3md.info -ref 2mdheat.rst7 -x mdcrd.3md
```

`3md.in` from tutorial:
```in
 &cntrl
  imin = 0, nstlim = 1000000, dt = 0.001,
  irest = 1, ntx = 5, ig = -1,
  temp0 = 298.0,
  ntc = 2, ntf = 2, tol = 0.00001,
  ntwx = 10000, ntwe = 0, ntwr = 1000, ntpr = 1000,
  cut = 8.0, iwrap = 0,
  ntt =3, gamma_ln=1.0, ntb = 2, ntp = 1, barostat = 2,
  nscm = 0,
  ntr=1, restraintmask=':1-81', restraint_wt=100.
  ioutfm=1, ntxo=2,
 /

```
Key settings:
```
imin = 0          This run is not a minimization run.
nstlim = 1000000  There will be 1000000 MD-steps. 
dt = 0.001        There is a time step of 1 fs. 
irest = 1         Restart the simulation from previously 
                    saved restart files.          
ntx = 5           The coordinates and velocities are 
                    read into the run.
ig = -1           The random seed number is based on the 
                    current date and time of the run.
temp0 = 298.0     The reference temperature is set to 298 K.
ntwx = 10000      The coordinates are written to a mdcrd file 10000 times. 
cut = 8.0         The non-bonded cutoff is 8.0 Angstroms.
ntt = 3           Langevin thermostat for temperature control is set.
gamma_ln=1.       The collision frequency gamma is set to 1 picosecond. 
ntb = 2           Constant pressure.
ntp = 1           Use isotropic position scaling.
barostat = 2      Use the Monte Carlo barostat. 
nscm = 0          No removal of center of mass motion.
ntr=1             Atoms which are restrained within the simulation.
restraintmask = ':1-81' Restrain solute (as in previous steps).
restraint_wt = 100  Positional restraint is 100 kcal/mol*Ang^-2 (same as above).
nmropt=1          NMR restraints and weight changes will be read.
ioutfm = 1        The format of the coordinate and velocity trajectory
                    files written as binary NetCDF.
ntxo=2            The "restrt" file format is NetCDF.
```

Change `restraintmask` (residues to be restrained) based on `amber_o98_wions_water.pdb` (294):
```in
&cntrl
  imin = 0, nstlim = 1000000, dt = 0.001,
  irest = 1, ntx = 5, ig = -1,
  temp0 = 298.0,
  ntc = 2, ntf = 2, tol = 0.00001,
  ntwx = 10000, ntwe = 0, ntwr = 1000, ntpr = 1000,
  cut = 8.0, iwrap = 0,
  ntt =3, gamma_ln=1.0, ntb = 2, ntp = 1, barostat = 2,
  nscm = 0,
  ntr=1, restraintmask=':1-294', restraint_wt=100.
  ioutfm=1, ntxo=2,
 /

```

## MD output

SUCCESS

Terminal:
```sh
Note: The following floating-point exceptions are signalling: IEEE_UNDERFLOW_FLAG IEEE_DENORMAL
```
`3md.out`
```out
------------------------------------------------------------------------------


      A V E R A G E S   O V E R    1000 S T E P S


 NSTEP =  1000000   TIME(PS) =    2000.000  TEMP(K) =   298.11  PRESS =     0.0
 Etot   =   -149136.8491  EKtot   =     26587.1084  EPtot      =   -175723.9575
 BOND   =       677.9804  ANGLE   =      1766.6180  DIHED      =      1380.1992
 UB     =         0.0000  IMP     =         0.0000  CMAP       =       747.9293
 1-4 NB =      1509.7030  1-4 EEL =     12185.6784  VDWAALS    =     21640.4741
 EELEC  =   -218096.2065  EHBOND  =         0.0000  RESTRAINT  =      2463.6667
 EAMBER (non-restraint)  =   -178187.6242
 EKCMT  =         0.0000  VIRIAL  =         0.0000  VOLUME     =    453923.1599
                                                    Density    =         0.9868
 ------------------------------------------------------------------------------


      R M S  F L U C T U A T I O N S


 NSTEP =  1000000   TIME(PS) =    2000.000  TEMP(K) =     1.44  PRESS =     0.0
 Etot   =       246.6849  EKtot   =       128.7856  EPtot      =       212.3141
 BOND   =        19.5979  ANGLE   =        25.5319  DIHED      =        10.2379
 UB     =         0.0000  IMP     =         0.0000  CMAP       =         4.9345
 1-4 NB =         8.4977  1-4 EEL =        18.1583  VDWAALS    =       218.3326
 EELEC  =       358.6147  EHBOND  =         0.0000  RESTRAINT  =        41.1056
 EAMBER (non-restraint)  =       171.2085
 EKCMT  =         0.0000  VIRIAL  =         0.0000  VOLUME     =      4691.4667
                                                    Density    =         0.0097
 ------------------------------------------------------------------------------

| MC Barostat:      10000 volume changes attempted.
| MC Barostat:       3027 changes successful ( 30.27%)
 ------------------------------------------------------------------------------

--------------------------------------------------------------------------------
   5.  TIMINGS
--------------------------------------------------------------------------------

|  NonSetup CPU Time in Major Routines:
|
|     Routine           Sec        %
|     ------------------------------
|     Nonbond          74.43    3.65
|     Bond              0.00    0.00
|     Angle             0.00    0.00
|     Dihedral          0.00    0.00
|     Shake             2.72    0.13
|     RunMD          1950.57   95.62
|     Other            12.10    0.59
|     ------------------------------
|     Total          2039.82

|  PME Nonbond Pairlist CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     Set Up Cit           0.00    0.00
|     Build List           0.00    0.00
|     ---------------------------------
|     Total                0.00    0.00

|  PME Direct Force CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     NonBonded Calc       0.00    0.00
|     Exclude Masked       0.00    0.00
|     Other                1.17    0.06
|     ---------------------------------
|     Total                1.17    0.06

|  PME Reciprocal Force CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     1D bspline           0.00    0.00
|     Grid Charges         0.00    0.00
|     Scalar Sum           0.00    0.00
|     Gradient Sum         0.00    0.00
|     FFT                  0.00    0.00
|     ---------------------------------
|     Total                0.00    0.00

|  Final Performance Info:
|     -----------------------------------------------------
|     Average timings for last   19000 steps:
|     Elapsed(s) =      38.48 Per Step(ms) =       2.03
|         ns/day =      42.66   seconds/ns =    2025.49
|
|     Average timings for all steps:
|     Elapsed(s) =    2041.35 Per Step(ms) =       2.04
|         ns/day =      42.32   seconds/ns =    2041.35
|     -----------------------------------------------------

|  Setup CPU time:            0.54 seconds
|  NonSetup CPU time:      2039.82 seconds
|  Total CPU time:         2040.35 seconds     0.57 hours

|  Setup wall time:           0    seconds
|  NonSetup wall time:     2042    seconds
|  Total wall time:        2042    seconds     0.57 hours
```

## Convert `parm7` and `rst7` to `pdb` for visualization

`3md_cpptraj_netCDF_to_pdb.out`
```out
	Reading 'amber_o98_ion.parm7' as Amber Topology
	Radius Set: modified Bondi radii (mbondi)
INPUT: Reading input from '3md_cpptraj_netCDF_to_pdb.in'
  [trajin 3md.rst7]
	Reading '3md.rst7' as Amber NC Restart
  [trajout 3md_rst7.pdb PDB]
	Writing '3md_rst7.pdb' as PDB
---------- RUN BEGIN -------------------------------------------------

PARAMETER FILES (1 total):
 0: amber_o98_ion.parm7, 56610 atoms, 13351 res, box: Truncated octahedron, 13059 mol, 12955 solvent

INPUT TRAJECTORIES (1 total):
 0: '3md.rst7' is a NetCDF AMBER restart file with coordinates, velocities, time, box, Parm amber_o98_ion.parm7 (Truncated octahedron box) (reading 1 of 1)
  Coordinate processing will occur on 1 frames.

OUTPUT TRAJECTORIES (1 total):
  '3md_rst7.pdb' (1 frames) is a PDB file

BEGIN TRAJECTORY PROCESSING:
Warning: No PDB space group specified.
Warning: Topology 'amber_o98_ion.parm7' has 12955 extra points
Warning:   that will not be included in output PDB.
Warning: To include them, specify 'include_ep'. Otherwise, use output PDB as
Warning:   topology or create a new topology with 'strip' or 'parmstrip'.
.....................................................
ACTIVE OUTPUT TRAJECTORIES (1):
  3md_rst7.pdb (coordinates, velocities, time, box)
----- 3md.rst7 (1-1, 1) -----
100% Complete.

Read 1 frames and processed 1 frames.
TIME: Avg. throughput= 8.4190 frames / second.

ACTION OUTPUT:
TIME: Analyses took 0.0000 seconds.

RUN TIMING:
TIME:		Init               : 0.0000 s (  0.01%)
TIME:		Trajectory Process : 0.1188 s ( 99.89%)
TIME:		Action Post        : 0.0000 s (  0.00%)
TIME:		Analysis           : 0.0000 s (  0.00%)
TIME:		Data File Write    : 0.0000 s (  0.00%)
TIME:		Other              : 0.0001 s (  0.00%)
TIME:	Run Total 0.1189 s
---------- RUN END ---------------------------------------------------
TIME: Total execution time: 0.2027 seconds.
--------------------------------------------------------------------------------
```

`3md_rst.pdb` on VMD\
![3md_rst_pdb_quicksurf](B2/trial5_tleap_script_parm7_rst7/3md_rst_pdb_quicksurf.png)\
It is "normal" for the water molecules to diffuse out, given the `3md.in` was run with `iwrap=0`,\
which does NOT re-frame the diffusing water molecules to the opposing face of the solvent box (use `iwrap=1` to do otherwise)\
`iwrap=0` may be better for diffusion coefficient calculations, since there is no water re-entering the system,\
but for visualization, `iwrap=1` is better

## Visualize with the raw `parm7` and `rst7` files without converting to `pdb` in the terminal

In [1]:
import parmed as pmd
import nglview as nv

Without any changes in representations

NOT GOOD: protein in ball and stick $\rightarrow$ hard to differentiate with water

In [7]:
# Load the topology and coordinates
structure = pmd.load_file('B2/trial5_tleap_script_parm7_rst7/amber_o98_ion.parm7', 'B2/trial5_tleap_script_parm7_rst7/3md.rst7')

# Create a NGLview widget
view = nv.show_parmed(structure)

# Display the widget
view

NGLWidget()

Show protein as cartoon and water as surface

NOT GOOD: water blocks proteins from view, even with very low opacity

In [13]:
# Create a NGLview widget
view1 = nv.show_parmed(structure)

# Clear default representation
view1.clear_representations()

# Add new representations
view1.add_cartoon('protein')
view1.add_surface('not protein', opacity=0.1)

# Display the widget
view1

NGLWidget()

Show protein as cartoon and water as translucent ball and stick

NOTE: canNOT tell accurately if solvent box has diffused out of original shape 

In [47]:
# Create a NGLview widget
view2 = nv.show_parmed(structure)

# Clear default representation
view2.clear_representations()

# Add new representations
view2.add_cartoon('protein')
view2.add_ball_and_stick('not protein', opacity=0.3)

# Display the widget
view2

NGLWidget()

# 4. Lower the restraints on the system

## MD input

Input files:
1. `4md.in`
2. `amber_o98_ion.parm7`
3. `3md.rst7`

Terminal command:
```sh
$AMBERHOME/bin/pmemd.cuda -O -i 4md.in -o 4md.out -p amber_o98_ion.parm7 -c 3md.rst7 -r 4md.rst7 -inf 4md.info -ref 3md.rst7 -x mdcrd.4md
```

`4md.in` from tutorial:
```in
 &cntrl
  imin = 0, nstlim = 1000000, dt = 0.001,
  irest = 1, ntx = 5, ig = -1,
  temp0 = 298.0,
  ntc = 2, ntf = 2, tol = 0.00001,
  ntwx = 10000, ntwe = 0, ntwr = 1000, ntpr = 1000,
  cut = 8.0, iwrap = 0,
  ntt =3,  gamma_ln=1.0, ntb = 2, ntp = 1,
  nscm = 0, barostat = 2,
  ntr=1, restraintmask=':1-81', restraint_wt=10.
  ioutfm=1, ntxo=2,
 /

```
Settings are all the same as `3md.in` except the `restraint_wt = 10`

Change `restraintmask` (residues to be restrained) based on `amber_o98_wions_water.pdb` (294):
```in
&cntrl
  imin = 0, nstlim = 1000000, dt = 0.001,
  irest = 1, ntx = 5, ig = -1,
  temp0 = 298.0,
  ntc = 2, ntf = 2, tol = 0.00001,
  ntwx = 10000, ntwe = 0, ntwr = 1000, ntpr = 1000,
  cut = 8.0, iwrap = 0,
  ntt =3,  gamma_ln=1.0, ntb = 2, ntp = 1,
  nscm = 0, barostat = 2,
  ntr=1, restraintmask=':1-294', restraint_wt=10.
  ioutfm=1, ntxo=2,
 /

```

## MD output

SUCCESS

Terminal:
```sh
Note: The following floating-point exceptions are signalling: IEEE_UNDERFLOW_FLAG IEEE_DENORMAL
```
`4md.out`
```out
 ------------------------------------------------------------------------------


      A V E R A G E S   O V E R    1000 S T E P S


 NSTEP =  1000000   TIME(PS) =    3000.000  TEMP(K) =   297.98  PRESS =     0.0
 Etot   =   -151245.6214  EKtot   =     26575.3464  EPtot      =   -177820.9678
 BOND   =       765.4776  ANGLE   =      2124.3182  DIHED      =      1503.0391
 UB     =         0.0000  IMP     =         0.0000  CMAP       =       745.7961
 1-4 NB =      1334.4957  1-4 EEL =     11816.1025  VDWAALS    =     21107.8569
 EELEC  =   -218633.5041  EHBOND  =         0.0000  RESTRAINT  =      1415.4502
 EAMBER (non-restraint)  =   -179236.4180
 EKCMT  =         0.0000  VIRIAL  =         0.0000  VOLUME     =    438669.3131
                                                    Density    =         1.0210
 ------------------------------------------------------------------------------


      R M S  F L U C T U A T I O N S


 NSTEP =  1000000   TIME(PS) =    3000.000  TEMP(K) =     1.38  PRESS =     0.0
 Etot   =       214.3780  EKtot   =       123.1727  EPtot      =       184.4337
 BOND   =        21.8147  ANGLE   =        33.0265  DIHED      =        15.0505
 UB     =         0.0000  IMP     =         0.0000  CMAP       =         7.1355
 1-4 NB =        12.0001  1-4 EEL =        28.1074  VDWAALS    =       214.8022
 EELEC  =       331.2786  EHBOND  =         0.0000  RESTRAINT  =        24.5898
 EAMBER (non-restraint)  =       159.8439
 EKCMT  =         0.0000  VIRIAL  =         0.0000  VOLUME     =      1111.6195
                                                    Density    =         0.0026
 ------------------------------------------------------------------------------

| MC Barostat:      10000 volume changes attempted.
| MC Barostat:       3037 changes successful ( 30.37%)
 ------------------------------------------------------------------------------

--------------------------------------------------------------------------------
   5.  TIMINGS
--------------------------------------------------------------------------------

|  NonSetup CPU Time in Major Routines:
|
|     Routine           Sec        %
|     ------------------------------
|     Nonbond          74.09    3.99
|     Bond              0.00    0.00
|     Angle             0.00    0.00
|     Dihedral          0.00    0.00
|     Shake             2.67    0.14
|     RunMD          1765.75   95.20
|     Other            12.35    0.67
|     ------------------------------
|     Total          1854.87

|  PME Nonbond Pairlist CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     Set Up Cit           0.00    0.00
|     Build List           0.00    0.00
|     ---------------------------------
|     Total                0.00    0.00

|  PME Direct Force CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     NonBonded Calc       0.00    0.00
|     Exclude Masked       0.00    0.00
|     Other                1.17    0.06
|     ---------------------------------
|     Total                1.17    0.06

|  PME Reciprocal Force CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     1D bspline           0.00    0.00
|     Grid Charges         0.00    0.00
|     Scalar Sum           0.00    0.00
|     Gradient Sum         0.00    0.00
|     FFT                  0.00    0.00
|     ---------------------------------
|     Total                0.00    0.00

|  Final Performance Info:
|     -----------------------------------------------------
|     Average timings for last   19000 steps:
|     Elapsed(s) =      34.99 Per Step(ms) =       1.84
|         ns/day =      46.91   seconds/ns =    1841.82
|
|     Average timings for all steps:
|     Elapsed(s) =    1855.15 Per Step(ms) =       1.86
|         ns/day =      46.57   seconds/ns =    1855.15
|     -----------------------------------------------------

|  Setup CPU time:            0.54 seconds
|  NonSetup CPU time:      1854.87 seconds
|  Total CPU time:         1855.41 seconds     0.52 hours

|  Setup wall time:           1    seconds
|  NonSetup wall time:     1855    seconds
|  Total wall time:        1856    seconds     0.52 hours
```

## Convert `parm7` and `rst7` to `pdb` for visualization

`4md_cpptraj_netCDF_to_pdb.out`
```out
	Reading 'amber_o98_ion.parm7' as Amber Topology
	Radius Set: modified Bondi radii (mbondi)
INPUT: Reading input from '4md_cpptraj_netCDF_to_pdb.in'
  [trajin 4md.rst7]
	Reading '4md.rst7' as Amber NC Restart
  [trajout 4md_rst7.pdb PDB]
	Writing '4md_rst7.pdb' as PDB
---------- RUN BEGIN -------------------------------------------------

PARAMETER FILES (1 total):
 0: amber_o98_ion.parm7, 56610 atoms, 13351 res, box: Truncated octahedron, 13059 mol, 12955 solvent

INPUT TRAJECTORIES (1 total):
 0: '4md.rst7' is a NetCDF AMBER restart file with coordinates, velocities, time, box, Parm amber_o98_ion.parm7 (Truncated octahedron box) (reading 1 of 1)
  Coordinate processing will occur on 1 frames.

OUTPUT TRAJECTORIES (1 total):
  '4md_rst7.pdb' (1 frames) is a PDB file

BEGIN TRAJECTORY PROCESSING:
Warning: No PDB space group specified.
Warning: Topology 'amber_o98_ion.parm7' has 12955 extra points
Warning:   that will not be included in output PDB.
Warning: To include them, specify 'include_ep'. Otherwise, use output PDB as
Warning:   topology or create a new topology with 'strip' or 'parmstrip'.
.....................................................
ACTIVE OUTPUT TRAJECTORIES (1):
  4md_rst7.pdb (coordinates, velocities, time, box)
----- 4md.rst7 (1-1, 1) -----
100% Complete.

Read 1 frames and processed 1 frames.
TIME: Avg. throughput= 4.4084 frames / second.

ACTION OUTPUT:
TIME: Analyses took 0.0000 seconds.

RUN TIMING:
TIME:		Init               : 0.0000 s (  0.01%)
TIME:		Trajectory Process : 0.2268 s ( 99.94%)
TIME:		Action Post        : 0.0000 s (  0.00%)
TIME:		Analysis           : 0.0000 s (  0.00%)
TIME:		Data File Write    : 0.0000 s (  0.00%)
TIME:		Other              : 0.0001 s (  0.00%)
TIME:	Run Total 0.2270 s
---------- RUN END ---------------------------------------------------
TIME: Total execution time: 0.4253 seconds.
--------------------------------------------------------------------------------
```

`4md_rst7.pdb` (Opaque, cyan) overlaid with `3md_rst7.pdb` (BrushedMetal, grey) on VMD\
![4md_3md_rst_pdb](B2/trial5_tleap_script_parm7_rst7/4md_3md_rst_pdb.png)\
There is slight deviation in the protein topology. Normal?

## Visualize overlaid `3md` and `4md` structures with `nglview`
FAIL

In [22]:
import parmed as pmd
import nglview as nv

In [42]:
# Load the structures
structure_3md = pmd.load_file('B2/trial5_tleap_script_parm7_rst7/amber_o98_ion.parm7', 
                              'B2/trial5_tleap_script_parm7_rst7/3md.rst7')
structure_4md = pmd.load_file('B2/trial5_tleap_script_parm7_rst7/amber_o98_ion.parm7', 
                              'B2/trial5_tleap_script_parm7_rst7/4md.rst7')

In [46]:
# Create a NGLview widget
view_4md = nv.NGLWidget()

# Add the structures to the widget
view_4md.add_trajectory(structure_4md)

# Clear default representations
view_4md.clear_representations()

# Add new representations for each structure
view_4md.add_cartoon('protein')

# Display the widget
view_4md

NGLWidget()

# 5. Minimize the system with restraints just on the backbone of the molecule
NOTE: NO longer restraining the residues AND backbone atoms are constant for all proteins\
$\rightarrow$ NO need to change (`restraintmask` in) `.in` from tutorial

## Minimization input

Input files:
1. `5min.in`
2. `amber_o98_ion.parm7`
3. `4md.rst7`

Terminal command:
```sh
$AMBERHOME/bin/pmemd -O -i 5min.in -o 5min.out -p amber_o98_ion.parm7 -c 4md.rst7 -r 5min.rst7 -inf 5min.info -ref 4md.rst7 -x mdcrd.5min
```

`5min.in` from tutorial:
```in
Minimization of everything excluding backbone
 &cntrl
  imin = 1, maxcyc = 1000,
  ncyc = 30, ntx = 1, 
  ntwe = 0, ntwr = 500, ntpr = 50,
  ntc = 2, ntf = 2, ntb = 1, ntp = 0,
  cut = 8.0,   
  ntr=1, restraintmask="@CA,N,C", restraint_wt=10.
  ioutfm=1, ntxo=2,
 /

```
NOTE: again, there is `ntc = 2, ntf = 2,`, which caused `1min.in` to fail (BUT NOT `heat` or `md` steps)\
$\rightarrow$ try minimizing with these parameters first and see if `5min.in` also fails

Key settings:
```
imin = 1        This run is a minimization run.
maxcyc = 1000   The maximum amount of minimization cycles is 1000.
ncyc = 30       The first 30 cycles will utilize the steepest descent 
                   algorithm before shifting to the conjugate gradient 
                   algorithm for the remaining cycles.
ntx = 1         The coordinates but not velocities are read formatted 
                   from the coordinate file provided. 
ntwe = 0        No mden files are written.
ntwr = 500      Number of steps "restrt" files are written.
ntpr = 50       Number steps in which "mdout" and "mdinfo" files are written.
ntr = 1         Use restraints 
restraintmask = "@CA,N,C"  This string specifies that alpha C, N, and 
                   carbons are all restrained.
restraint_wt = 10  Weight of the positional restraint is 10 kcal/mol*Ang^-2. 
```

## Minimization output

SUCCESS(?)

NOTE: NO terminal output of `IEEE_UNDERFLOW_FLAG IEEE_DENORMAL`\
$\rightarrow$ really seems like it is an issue with `pmemd.cuda` and Fortran versions

`5min.out`
```out
     .... RESTARTED DUE TO LINMIN FAILURE ...


   NSTEP       ENERGY          RMS            GMAX         NAME    NUMBER
   1000      -1.8515E+05     2.1938E+01     7.9868E+01     O       17631

 BOND    =      158.9485  ANGLE   =      983.9760  DIHED      =     1324.7542
 UB      =        0.0000  IMP     =        0.0000  CMAP       =      740.1301
 VDWAALS =    16743.4875  EEL     =  -218151.7843  HBOND      =        0.0000
 1-4 VDW =     1240.6372  1-4 EEL =    11798.2466  RESTRAINT  =       13.2670
 EAMBER  =  -185161.6042


  Maximum number of minimization cycles reached.


                    FINAL RESULTS



   NSTEP       ENERGY          RMS            GMAX         NAME    NUMBER
   1000      -1.8515E+05     2.1938E+01     7.9868E+01     O       17631

 BOND    =      158.9485  ANGLE   =      983.9760  DIHED      =     1324.7542
 UB      =        0.0000  IMP     =        0.0000  CMAP       =      740.1301
 VDWAALS =    16743.4875  EEL     =  -218151.7843  HBOND      =        0.0000
 1-4 VDW =     1240.6372  1-4 EEL =    11798.2466  RESTRAINT  =       13.2670
 EAMBER  =  -185161.6042
--------------------------------------------------------------------------------
   5.  TIMINGS
--------------------------------------------------------------------------------

|  NonSetup CPU Time in Major Routines:
|
|     Routine           Sec        %
|     ------------------------------
|     Nonbond         160.79   97.52
|     Bond              0.03    0.02
|     Angle             0.36    0.22
|     Dihedral          2.17    1.31
|     Shake             0.97    0.59
|     Other             0.56    0.34
|     ------------------------------
|     Total           164.87

|  PME Nonbond Pairlist CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     Set Up Cit           0.00    0.00
|     Build List           0.69    0.42
|     ---------------------------------
|     Total                0.69    0.42

|  PME Direct Force CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     NonBonded Calc     125.87   76.34
|     Exclude Masked       2.05    1.25
|     Other                0.51    0.31
|     ---------------------------------
|     Total              128.43   77.90

|  PME Reciprocal Force CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     1D bspline           1.87    1.14
|     Grid Charges         2.42    1.47
|     Scalar Sum           9.37    5.68
|     Gradient Sum         4.58    2.78
|     FFT                 13.02    7.89
|     ---------------------------------
|     Total               31.26   18.96

|  Setup CPU time:            0.29 seconds
|  NonSetup CPU time:       164.87 seconds
|  Total CPU time:          165.16 seconds     0.05 hours

|  Setup wall time:           1    seconds
|  NonSetup wall time:      164    seconds
|  Total wall time:         165    seconds     0.05 hours
```

`LINMIN FAILURE` https://ambermd.org/Questions/linmin.html
```
From: dap@portal.vpharm.com (David A. Pearlman)
Subject: LINMIN and SHAKE failures
Date: Fri, 18 Mar 1994 16:21:33 -0500 (EST)
```
The LINMIN failures occur frequently when performing minimization. They don't mean that any sort of evil failure of minimization has occurred, only that the minimizer got "stuck" in a place from which the minimization algorithm could not find a way out. Unless there is something very askew with your system, the amount of minimization that has occurred by the time you reach such a "sticking" point will be sufficient to move on to MD or Gibbs. (It wouldn't be sufficient to carry out a normal modes calculation, but a different minimizer is used then, anyway).

SHAKE has a propensity to fail when there are very large forces in the system. If your system is poorly defined at the onset, it is likely that it will contain a number of hot spots, where the initial forces will be very large. Attempting to use SHAKE at this point increases the chance that you will encounter a "fatal error" due to SHAKE. In general, it is not necesary to use SHAKE during minimization, even if you will subsequently be using SHAKE in MD or GIBBS. But if you want to use SHAKE, I would suggest that you first minimize the system without SHAKE (to remove any extreme hot spots), then run a second minimization with SHAKE on, starting with the coordinates from the first minimization. This 2-step process has a greater likelihood of working. Although, as noted before, I don't believe there is much value in minimizing with SHAKE on for most cases... 

## Convert `parm7` and `rst7` to `pdb` for visualization

`5min_cpptraj_netCDF_to_pdb.out`
```out
	Reading 'amber_o98_ion.parm7' as Amber Topology
	Radius Set: modified Bondi radii (mbondi)
INPUT: Reading input from '5min_cpptraj_netCDF_to_pdb.in'
  [trajin 5min.rst7]
	Reading '5min.rst7' as Amber NC Restart
  [trajout 5min_rst7.pdb PDB]
	Writing '5min_rst7.pdb' as PDB
---------- RUN BEGIN -------------------------------------------------

PARAMETER FILES (1 total):
 0: amber_o98_ion.parm7, 56610 atoms, 13351 res, box: Truncated octahedron, 13059 mol, 12955 solvent

INPUT TRAJECTORIES (1 total):
 0: '5min.rst7' is a NetCDF AMBER restart file with coordinates, time, box, Parm amber_o98_ion.parm7 (Truncated octahedron box) (reading 1 of 1)
  Coordinate processing will occur on 1 frames.

OUTPUT TRAJECTORIES (1 total):
  '5min_rst7.pdb' (1 frames) is a PDB file

BEGIN TRAJECTORY PROCESSING:
Warning: No PDB space group specified.
Warning: Topology 'amber_o98_ion.parm7' has 12955 extra points
Warning:   that will not be included in output PDB.
Warning: To include them, specify 'include_ep'. Otherwise, use output PDB as
Warning:   topology or create a new topology with 'strip' or 'parmstrip'.
.....................................................
ACTIVE OUTPUT TRAJECTORIES (1):
  5min_rst7.pdb (coordinates, time, box)
----- 5min.rst7 (1-1, 1) -----
100% Complete.

Read 1 frames and processed 1 frames.
TIME: Avg. throughput= 6.7099 frames / second.

ACTION OUTPUT:
TIME: Analyses took 0.0000 seconds.

RUN TIMING:
TIME:		Init               : 0.0000 s (  0.00%)
TIME:		Trajectory Process : 0.1490 s ( 99.96%)
TIME:		Action Post        : 0.0000 s (  0.00%)
TIME:		Analysis           : 0.0000 s (  0.00%)
TIME:		Data File Write    : 0.0000 s (  0.00%)
TIME:		Other              : 0.0001 s (  0.00%)
TIME:	Run Total 0.1491 s
---------- RUN END ---------------------------------------------------
TIME: Total execution time: 0.2292 seconds.
--------------------------------------------------------------------------------
```

`5min_rst7.pdb` (Opaque, cyan) overlaid with `4md_rst7.pdb` (BrushedMetal, grey) on VMD\
![4md_5min_rst_pdb_quicksurf](B2/trial5_tleap_script_parm7_rst7/4md_5min_rst_pdb_quicksurf.png)\
Seems like minimization does NOT alter protein topology (complete overlap)?

# 6. Relax system at a constant pressure with backbone restraints

## MD input

Input files:
1. `6md.in`
2. `amber_o98_ion.parm7`
3. `5min.rst7`

Terminal command:
```sh
$AMBERHOME/bin/pmemd.cuda -O -i 6md.in -o 6md.out -p amber_o98_ion.parm7 -c 5min.rst7 -r 6md.rst7 -inf 6md.info -ref 5min.rst7 -x mdcrd.6md
```

`6md.in` from tutorial:
```in
&cntrl
  imin = 0, nstlim = 1000000, dt = 0.001,
  irest = 0, ntx = 1, ig = -1,
  tempi = 298.0, temp0 = 298.0,
  ntc = 2, ntf = 2, tol = 0.00001,
  ntwx = 10000, ntwe = 0, ntwr = 1000, ntpr = 1000,
  cut = 8.0, iwrap = 0,
  ntt =3, gamma_ln=1.0, ntb = 2, ntp = 1, 
  nscm = 0, barostat = 2,
  ntr=1, restraintmask="@CA,N,C", restraint_wt=10.
  ioutfm=1, ntxo=2,
 /

```
Key settings:\
Settings are the same as `4md.in` except:
```
restraintmask = "@CA,N,C"  This string specifies that alpha C, N, and 
                   carbons are all restrained.
restraint_wt = 10  Weight of the positional restraint is 10 kcal/mol*Ang^-2.
```

## MD output

SUCCESS

Terminal:
```sh
Note: The following floating-point exceptions are signalling: IEEE_UNDERFLOW_FLAG IEEE_DENORMAL
```
`6md.out`
```out
 ------------------------------------------------------------------------------


      A V E R A G E S   O V E R    1000 S T E P S


 NSTEP =  1000000   TIME(PS) =    1000.000  TEMP(K) =   297.89  PRESS =     0.0
 Etot   =   -153862.3551  EKtot   =     26567.1586  EPtot      =   -180429.5137
 BOND   =       831.7841  ANGLE   =      2265.4331  DIHED      =      1611.9542
 UB     =         0.0000  IMP     =         0.0000  CMAP       =       666.4837
 1-4 NB =      1133.8010  1-4 EEL =     11521.4713  VDWAALS    =     20460.0354
 EELEC  =   -219163.4174  EHBOND  =         0.0000  RESTRAINT  =       242.9410
 EAMBER (non-restraint)  =   -180672.4546
 EKCMT  =         0.0000  VIRIAL  =         0.0000  VOLUME     =    430648.7878
                                                    Density    =         1.0400
 ------------------------------------------------------------------------------


      R M S  F L U C T U A T I O N S


 NSTEP =  1000000   TIME(PS) =    1000.000  TEMP(K) =     1.38  PRESS =     0.0
 Etot   =       237.4818  EKtot   =       123.3627  EPtot      =       202.4619
 BOND   =        23.5575  ANGLE   =        37.8247  DIHED      =        23.6903
 UB     =         0.0000  IMP     =         0.0000  CMAP       =        10.7109
 1-4 NB =        13.2879  1-4 EEL =        39.8109  VDWAALS    =       217.9692
 EELEC  =       332.8790  EHBOND  =         0.0000  RESTRAINT  =         9.9688
 EAMBER (non-restraint)  =       192.4931
 EKCMT  =         0.0000  VIRIAL  =         0.0000  VOLUME     =       960.2370
                                                    Density    =         0.0023
 ------------------------------------------------------------------------------

| MC Barostat:      10000 volume changes attempted.
| MC Barostat:       3385 changes successful ( 33.85%)
 ------------------------------------------------------------------------------

--------------------------------------------------------------------------------
   5.  TIMINGS
--------------------------------------------------------------------------------

|  NonSetup CPU Time in Major Routines:
|
|     Routine           Sec        %
|     ------------------------------
|     Nonbond          75.60    4.09
|     Bond              0.00    0.00
|     Angle             0.00    0.00
|     Dihedral          0.00    0.00
|     Shake             2.68    0.14
|     RunMD          1756.85   95.10
|     Other            12.33    0.67
|     ------------------------------
|     Total          1847.46

|  PME Nonbond Pairlist CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     Set Up Cit           0.00    0.00
|     Build List           0.00    0.00
|     ---------------------------------
|     Total                0.00    0.00

|  PME Direct Force CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     NonBonded Calc       0.00    0.00
|     Exclude Masked       0.00    0.00
|     Other                1.17    0.06
|     ---------------------------------
|     Total                1.17    0.06

|  PME Reciprocal Force CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     1D bspline           0.00    0.00
|     Grid Charges         0.00    0.00
|     Scalar Sum           0.00    0.00
|     Gradient Sum         0.00    0.00
|     FFT                  0.00    0.00
|     ---------------------------------
|     Total                0.00    0.00

|  Final Performance Info:
|     -----------------------------------------------------
|     Average timings for last   16000 steps:
|     Elapsed(s) =      29.91 Per Step(ms) =       1.87
|         ns/day =      46.22   seconds/ns =    1869.22
|
|     Average timings for all steps:
|     Elapsed(s) =    1848.86 Per Step(ms) =       1.85
|         ns/day =      46.73   seconds/ns =    1848.86
|     -----------------------------------------------------

|  Setup CPU time:            0.53 seconds
|  NonSetup CPU time:      1847.46 seconds
|  Total CPU time:         1847.99 seconds     0.51 hours

|  Setup wall time:           1    seconds
|  NonSetup wall time:     1849    seconds
|  Total wall time:        1850    seconds     0.51 hours
```

## Convert `parm7` and `rst7` to `pdb` for visualization

`6md_cpptraj_netCDF_to_pdb.out`
```out
	Reading 'amber_o98_ion.parm7' as Amber Topology
	Radius Set: modified Bondi radii (mbondi)
INPUT: Reading input from '6md_cpptraj_netCDF_to_pdb.in'
  [trajin 6md.rst7]
	Reading '6md.rst7' as Amber NC Restart
  [trajout 6md_rst7.pdb PDB]
	Writing '6md_rst7.pdb' as PDB
---------- RUN BEGIN -------------------------------------------------

PARAMETER FILES (1 total):
 0: amber_o98_ion.parm7, 56610 atoms, 13351 res, box: Truncated octahedron, 13059 mol, 12955 solvent

INPUT TRAJECTORIES (1 total):
 0: '6md.rst7' is a NetCDF AMBER restart file with coordinates, velocities, time, box, Parm amber_o98_ion.parm7 (Truncated octahedron box) (reading 1 of 1)
  Coordinate processing will occur on 1 frames.

OUTPUT TRAJECTORIES (1 total):
  '6md_rst7.pdb' (1 frames) is a PDB file

BEGIN TRAJECTORY PROCESSING:
Warning: No PDB space group specified.
Warning: Topology 'amber_o98_ion.parm7' has 12955 extra points
Warning:   that will not be included in output PDB.
Warning: To include them, specify 'include_ep'. Otherwise, use output PDB as
Warning:   topology or create a new topology with 'strip' or 'parmstrip'.
.....................................................
ACTIVE OUTPUT TRAJECTORIES (1):
  6md_rst7.pdb (coordinates, velocities, time, box)
----- 6md.rst7 (1-1, 1) -----
100% Complete.

Read 1 frames and processed 1 frames.
TIME: Avg. throughput= 9.0232 frames / second.

ACTION OUTPUT:
TIME: Analyses took 0.0000 seconds.

RUN TIMING:
TIME:		Init               : 0.0000 s (  0.01%)
TIME:		Trajectory Process : 0.1108 s ( 99.89%)
TIME:		Action Post        : 0.0000 s (  0.00%)
TIME:		Analysis           : 0.0000 s (  0.00%)
TIME:		Data File Write    : 0.0000 s (  0.00%)
TIME:		Other              : 0.0001 s (  0.00%)
TIME:	Run Total 0.1110 s
---------- RUN END ---------------------------------------------------
TIME: Total execution time: 0.2203 seconds.
--------------------------------------------------------------------------------
```

`6md_rst7.pdb` (Opaque, cyan) overlaid with `5min_rst7.pdb` (BrushedMetal, grey) on VMD\
![6md_5min_rst_pdb](B2/trial5_tleap_script_parm7_rst7/6md_5min_rst_pdb.png)

# 7. Reduce the backbone restraint force

## MD input

Input files:
1. `7md.in`
2. `amber_o98_ion.parm7`
3. `6md.rst7`

Terminal command:
```sh
$AMBERHOME/bin/pmemd.cuda -O -i 7md.in -o 7md.out -p amber_o98_ion.parm7 -c 6md.rst7 -r 7md.rst7 -inf 7md.info -ref 6md.rst7 -x mdcrd.7md
```

`7md.in` from tutorial:
```in
&cntrl
  imin = 0, nstlim = 1000000, dt = 0.001,
  irest = 1, ntx = 5, ig = -1,
  temp0 = 298.0,
  ntc = 2, ntf = 2, tol = 0.00001,
  ntwx = 10000, ntwe = 0, ntwr = 1000, ntpr = 1000,
  cut = 8.0, iwrap = 0,
  ntt =3, gamma_ln=1.0, ntb = 2, ntp = 1,
  nscm = 0, barostat = 2,
  ntr=1, restraintmask="@CA,N,C", restraint_wt=1.
  ioutfm=1, ntxo=2,
 /

```
Key settings:\
Settings are the same as `6md.in` except:
```
restraint_wt = 1  Weight of the positional restraint is 1 kcal/mol*Ang^-2.
```

## MD output

SUCCESS

Terminal:
```sh
Note: The following floating-point exceptions are signalling: IEEE_UNDERFLOW_FLAG IEEE_DENORMAL
```
`7md.out`
```out
 ------------------------------------------------------------------------------


      A V E R A G E S   O V E R    1000 S T E P S


 NSTEP =  1000000   TIME(PS) =    2000.000  TEMP(K) =   297.91  PRESS =     0.0
 Etot   =   -154143.3494  EKtot   =     26569.3403  EPtot      =   -180712.6897
 BOND   =       853.8209  ANGLE   =      2291.4862  DIHED      =      1605.7779
 UB     =         0.0000  IMP     =         0.0000  CMAP       =       603.7702
 1-4 NB =      1108.6484  1-4 EEL =     11422.8989  VDWAALS    =     20473.8392
 EELEC  =   -219166.6181  EHBOND  =         0.0000  RESTRAINT  =        93.6865
 EAMBER (non-restraint)  =   -180806.3762
 EKCMT  =         0.0000  VIRIAL  =         0.0000  VOLUME     =    430239.4044
                                                    Density    =         1.0410
 ------------------------------------------------------------------------------


      R M S  F L U C T U A T I O N S


 NSTEP =  1000000   TIME(PS) =    2000.000  TEMP(K) =     1.44  PRESS =     0.0
 Etot   =       226.9442  EKtot   =       128.8065  EPtot      =       186.9791
 BOND   =        25.0012  ANGLE   =        35.3035  DIHED      =        20.5987
 UB     =         0.0000  IMP     =         0.0000  CMAP       =        12.6430
 1-4 NB =        13.9563  1-4 EEL =        39.1783  VDWAALS    =       222.9388
 EELEC  =       335.5441  EHBOND  =         0.0000  RESTRAINT  =         6.1516
 EAMBER (non-restraint)  =       180.8274
 EKCMT  =         0.0000  VIRIAL  =         0.0000  VOLUME     =       875.6214
                                                    Density    =         0.0021
 ------------------------------------------------------------------------------

| MC Barostat:      10000 volume changes attempted.
| MC Barostat:       2835 changes successful ( 28.35%)
 ------------------------------------------------------------------------------

--------------------------------------------------------------------------------
   5.  TIMINGS
--------------------------------------------------------------------------------

|  NonSetup CPU Time in Major Routines:
|
|     Routine           Sec        %
|     ------------------------------
|     Nonbond          76.39    4.13
|     Bond              0.00    0.00
|     Angle             0.00    0.00
|     Dihedral          0.00    0.00
|     Shake             2.76    0.15
|     RunMD          1759.10   95.05
|     Other            12.43    0.67
|     ------------------------------
|     Total          1850.68

|  PME Nonbond Pairlist CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     Set Up Cit           0.00    0.00
|     Build List           0.00    0.00
|     ---------------------------------
|     Total                0.00    0.00

|  PME Direct Force CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     NonBonded Calc       0.00    0.00
|     Exclude Masked       0.00    0.00
|     Other                1.17    0.06
|     ---------------------------------
|     Total                1.17    0.06

|  PME Reciprocal Force CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     1D bspline           0.00    0.00
|     Grid Charges         0.00    0.00
|     Scalar Sum           0.00    0.00
|     Gradient Sum         0.00    0.00
|     FFT                  0.00    0.00
|     ---------------------------------
|     Total                0.00    0.00

|  Final Performance Info:
|     -----------------------------------------------------
|     Average timings for last   17000 steps:
|     Elapsed(s) =      31.52 Per Step(ms) =       1.85
|         ns/day =      46.60   seconds/ns =    1853.98
|
|     Average timings for all steps:
|     Elapsed(s) =    1850.66 Per Step(ms) =       1.85
|         ns/day =      46.69   seconds/ns =    1850.66
|     -----------------------------------------------------

|  Setup CPU time:            0.55 seconds
|  NonSetup CPU time:      1850.68 seconds
|  Total CPU time:         1851.23 seconds     0.51 hours

|  Setup wall time:           1    seconds
|  NonSetup wall time:     1851    seconds
|  Total wall time:        1852    seconds     0.51 hours
```

## Convert `parm7` and `rst7` to `pdb` for visualization

`7md_cpptraj_netCDF_to_pdb.out`
```out
	Reading 'amber_o98_ion.parm7' as Amber Topology
	Radius Set: modified Bondi radii (mbondi)
INPUT: Reading input from '7md_cpptraj_netCDF_to_pdb.in'
  [trajin 7md.rst7]
	Reading '7md.rst7' as Amber NC Restart
  [trajout 7md_rst7.pdb PDB]
	Writing '7md_rst7.pdb' as PDB
---------- RUN BEGIN -------------------------------------------------

PARAMETER FILES (1 total):
 0: amber_o98_ion.parm7, 56610 atoms, 13351 res, box: Truncated octahedron, 13059 mol, 12955 solvent

INPUT TRAJECTORIES (1 total):
 0: '7md.rst7' is a NetCDF AMBER restart file with coordinates, velocities, time, box, Parm amber_o98_ion.parm7 (Truncated octahedron box) (reading 1 of 1)
  Coordinate processing will occur on 1 frames.

OUTPUT TRAJECTORIES (1 total):
  '7md_rst7.pdb' (1 frames) is a PDB file

BEGIN TRAJECTORY PROCESSING:
Warning: No PDB space group specified.
Warning: Topology 'amber_o98_ion.parm7' has 12955 extra points
Warning:   that will not be included in output PDB.
Warning: To include them, specify 'include_ep'. Otherwise, use output PDB as
Warning:   topology or create a new topology with 'strip' or 'parmstrip'.
.....................................................
ACTIVE OUTPUT TRAJECTORIES (1):
  7md_rst7.pdb (coordinates, velocities, time, box)
----- 7md.rst7 (1-1, 1) -----
100% Complete.

Read 1 frames and processed 1 frames.
TIME: Avg. throughput= 1.6166 frames / second.

ACTION OUTPUT:
TIME: Analyses took 0.0000 seconds.

RUN TIMING:
TIME:		Init               : 0.0000 s (  0.00%)
TIME:		Trajectory Process : 0.6186 s ( 99.98%)
TIME:		Action Post        : 0.0000 s (  0.00%)
TIME:		Analysis           : 0.0000 s (  0.00%)
TIME:		Data File Write    : 0.0000 s (  0.00%)
TIME:		Other              : 0.0001 s (  0.00%)
TIME:	Run Total 0.6187 s
---------- RUN END ---------------------------------------------------
TIME: Total execution time: 0.9099 seconds.
--------------------------------------------------------------------------------
```
Took much (4.5 times) longer to generate PDB this time?

`7md_rst7.pdb` (Opaque, cyan) overlaid with `6md_rst7.pdb` (BrushedMetal, grey) on VMD\
![7md_6md_rst_pdb](B2/trial5_tleap_script_parm7_rst7/7md_6md_rst_pdb.png)\
But there is no observable difference from `5min_rst7.pdb`...?\
Maybe slightly more deviation...?

# 8. Continue to reduce the backbone restraint force

## MD input

Input files:
1. `8md.in`
2. `amber_o98_ion.parm7`
3. `7md.rst7`

Terminal command:
```sh
$AMBERHOME/bin/pmemd.cuda -O -i 8md.in -o 8md.out -p amber_o98_ion.parm7 -c 7md.rst7 -r 8md.rst7 -inf 8md.info -ref 7md.rst7 -x mdcrd.8md
```

`8md.in` from tutorial:
```in
&cntrl
  imin = 0, nstlim = 1000000, dt = 0.001,
  irest = 1, ntx = 5, ig = -1,
  temp0 = 298.0,
  ntc = 2, ntf = 2, tol = 0.00001,
  ntwx = 10000, ntwe = 0, ntwr = 1000, ntpr = 1000,
  cut = 8.0, iwrap = 0,
  ntt =3, gamma_ln=1.0, ntb = 2, ntp = 1,
  nscm = 0, barostat = 2,
  ntr=1, restraintmask="@CA,N,C", restraint_wt=0.1
  ioutfm=1, ntxo=2,
 /

```
Key settings:\
Settings are the same as `7md.in` except:
```
restraint_wt = 0.1  Weight of the positional restraint is 0.1 kcal/mol*Ang^-2.
```

## MD output

SUCCESS

Terminal:
```sh
Note: The following floating-point exceptions are signalling: IEEE_UNDERFLOW_FLAG IEEE_DENORMAL
```
`8md.out`
```out
 ------------------------------------------------------------------------------


      A V E R A G E S   O V E R    1000 S T E P S


 NSTEP =  1000000   TIME(PS) =    3000.000  TEMP(K) =   298.06  PRESS =     0.0
 Etot   =   -154212.2321  EKtot   =     26582.7179  EPtot      =   -180794.9500
 BOND   =       863.0641  ANGLE   =      2302.3070  DIHED      =      1603.0023
 UB     =         0.0000  IMP     =         0.0000  CMAP       =       547.6640
 1-4 NB =      1105.3952  1-4 EEL =     11381.9292  VDWAALS    =     20486.0013
 EELEC  =   -219121.2002  EHBOND  =         0.0000  RESTRAINT  =        36.8872
 EAMBER (non-restraint)  =   -180831.8372
 EKCMT  =         0.0000  VIRIAL  =         0.0000  VOLUME     =    430132.5011
                                                    Density    =         1.0413
 ------------------------------------------------------------------------------


      R M S  F L U C T U A T I O N S


 NSTEP =  1000000   TIME(PS) =    3000.000  TEMP(K) =     1.40  PRESS =     0.0
 Etot   =       228.2284  EKtot   =       124.5770  EPtot      =       186.8396
 BOND   =        23.9912  ANGLE   =        37.8279  DIHED      =        21.0171
 UB     =         0.0000  IMP     =         0.0000  CMAP       =        14.7434
 1-4 NB =        14.0991  1-4 EEL =        40.5401  VDWAALS    =       229.5753
 EELEC  =       337.3713  EHBOND  =         0.0000  RESTRAINT  =         4.9893
 EAMBER (non-restraint)  =       181.8503
 EKCMT  =         0.0000  VIRIAL  =         0.0000  VOLUME     =       832.2020
                                                    Density    =         0.0020
 ------------------------------------------------------------------------------

| MC Barostat:      10000 volume changes attempted.
| MC Barostat:       2807 changes successful ( 28.07%)
 ------------------------------------------------------------------------------

--------------------------------------------------------------------------------
   5.  TIMINGS
--------------------------------------------------------------------------------

|  NonSetup CPU Time in Major Routines:
|
|     Routine           Sec        %
|     ------------------------------
|     Nonbond          75.99    4.11
|     Bond              0.00    0.00
|     Angle             0.00    0.00
|     Dihedral          0.00    0.00
|     Shake             2.70    0.15
|     RunMD          1759.65   95.08
|     Other            12.37    0.67
|     ------------------------------
|     Total          1850.71

|  PME Nonbond Pairlist CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     Set Up Cit           0.00    0.00
|     Build List           0.00    0.00
|     ---------------------------------
|     Total                0.00    0.00

|  PME Direct Force CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     NonBonded Calc       0.00    0.00
|     Exclude Masked       0.00    0.00
|     Other                1.16    0.06
|     ---------------------------------
|     Total                1.16    0.06

|  PME Reciprocal Force CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     1D bspline           0.00    0.00
|     Grid Charges         0.00    0.00
|     Scalar Sum           0.00    0.00
|     Gradient Sum         0.00    0.00
|     FFT                  0.00    0.00
|     ---------------------------------
|     Total                0.00    0.00

|  Final Performance Info:
|     -----------------------------------------------------
|     Average timings for last   18000 steps:
|     Elapsed(s) =      33.42 Per Step(ms) =       1.86
|         ns/day =      46.53   seconds/ns =    1856.85
|
|     Average timings for all steps:
|     Elapsed(s) =    1850.83 Per Step(ms) =       1.85
|         ns/day =      46.68   seconds/ns =    1850.83
|     -----------------------------------------------------

|  Setup CPU time:            0.53 seconds
|  NonSetup CPU time:      1850.71 seconds
|  Total CPU time:         1851.23 seconds     0.51 hours

|  Setup wall time:           1    seconds
|  NonSetup wall time:     1851    seconds
|  Total wall time:        1852    seconds     0.51 hours
```

## Convert `parm7` and `rst7` to `pdb` for visualization

`8md_cpptraj_netCDF_to_pdb.out`
```out
	Reading 'amber_o98_ion.parm7' as Amber Topology
	Radius Set: modified Bondi radii (mbondi)
INPUT: Reading input from '8md_cpptraj_netCDF_to_pdb.in'
  [trajin 8md.rst7]
	Reading '8md.rst7' as Amber NC Restart
  [trajout 8md_rst7.pdb PDB]
	Writing '8md_rst7.pdb' as PDB
---------- RUN BEGIN -------------------------------------------------

PARAMETER FILES (1 total):
 0: amber_o98_ion.parm7, 56610 atoms, 13351 res, box: Truncated octahedron, 13059 mol, 12955 solvent

INPUT TRAJECTORIES (1 total):
 0: '8md.rst7' is a NetCDF AMBER restart file with coordinates, velocities, time, box, Parm amber_o98_ion.parm7 (Truncated octahedron box) (reading 1 of 1)
  Coordinate processing will occur on 1 frames.

OUTPUT TRAJECTORIES (1 total):
  '8md_rst7.pdb' (1 frames) is a PDB file

BEGIN TRAJECTORY PROCESSING:
Warning: No PDB space group specified.
Warning: Topology 'amber_o98_ion.parm7' has 12955 extra points
Warning:   that will not be included in output PDB.
Warning: To include them, specify 'include_ep'. Otherwise, use output PDB as
Warning:   topology or create a new topology with 'strip' or 'parmstrip'.
.....................................................
ACTIVE OUTPUT TRAJECTORIES (1):
  8md_rst7.pdb (coordinates, velocities, time, box)
----- 8md.rst7 (1-1, 1) -----
100% Complete.

Read 1 frames and processed 1 frames.
TIME: Avg. throughput= 7.3935 frames / second.

ACTION OUTPUT:
TIME: Analyses took 0.0000 seconds.

RUN TIMING:
TIME:		Init               : 0.0000 s (  0.02%)
TIME:		Trajectory Process : 0.1353 s ( 99.88%)
TIME:		Action Post        : 0.0000 s (  0.00%)
TIME:		Analysis           : 0.0000 s (  0.00%)
TIME:		Data File Write    : 0.0000 s (  0.00%)
TIME:		Other              : 0.0001 s (  0.00%)
TIME:	Run Total 0.1354 s
---------- RUN END ---------------------------------------------------
TIME: Total execution time: 0.2191 seconds.
--------------------------------------------------------------------------------
```

`8md_rst7.pdb` (Opaque, cyan) overlaid with `7md_rst7.pdb` (BrushedMetal, grey) on VMD\
![8md_7md_rst_pdb](B2/trial5_tleap_script_parm7_rst7/8md_7md_rst_pdb.png)

# 9. Relax the system with no restraints

## MD input

Input files:
1. `9md.in`
2. `amber_o98_ion.parm7`
3. `8md.rst7`

Terminal command:
```sh
$AMBERHOME/bin/pmemd.cuda -O -i 9md.in -o 9md.out -p amber_o98_ion.parm7 -c 8md.rst7 -r 9md.rst7 -inf 9md.info -ref 8md.rst7 -x mdcrd.9md
```

`9md.in` from tutorial:
```in
&cntrl   
  imin = 0, nstlim = 1000000, dt = 0.001,
  irest = 1, ntx = 5, ig = -1,
  temp0 = 298.0,
  ntc = 2, ntf = 2, tol = 0.00001,
  ntwx = 10000, ntwe = 0, ntwr = 1000, ntpr = 1000,
  cut = 8.0, iwrap = 0,
  ntt =3, gamma_ln=1.0, ntb = 2, ntp = 1,
  nscm = 1000, barostat = 2,
  ioutfm=1, ntxo=2,
 /

```
NOTE: time step is still small (1 fs) since the system is still relaxing and the translation and rotational COM motion is removed every 1000 steps, and that there are no restraints on the system anymore. 

Key settings:
```
imin = 0         This run is not a minimization run.
nstlim = 1000000 There will be 1000000 MD-steps. 
dt = 0.001       There is a time step of 1 femtosecond. 
ntx = 5          The coordinates and velocities are read into the run.
ig = -1          The random seed number based on the current 
                     date and time of the run.
temp0 = 298.0    Reference temperature is 298 K.
ntc = 2          Bonds involving H are constrained. 
ntf = 2          Bonds involving H are omitted from force evaluation. 
tol = 0.00001    The error of tolerance is 0.00001 Angstroms.
ntwx = 10000     The coordinates are written to a mdcrd file 10000 times. 
ntwe = 0         No mden files are written.
ntwr = 1000      Number of steps "restrt" files are written. 
ntpr = 1000      Number of steps "mdout" and "mdinfo" files are written.
cut = 8.0        The non-bonded cutoff is 8.0 Angstroms.
iwrap = 0        No wrapping is performed. 
ntt = 3          Langevin thermostat for temperature control is set.
gamma_ln=1.      The collision frequency gamma is set to 1 picosecond. 
ntb = 2          There is constant pressure.
ntp = 1          The md runs with isotropic position scaling.
nscm = 1000      Removal of translation and rotational COM motion 
                    at every 1000 steps.
barostat = 2     This simulation uses the Monte Carlo barostat.
ioutfm = 1        The format of the coordinate and velocity trajectory
                    files written as binary NetCDF.
ntxo=2            The "restrt" file format is NetCDF.
```

## MD output

SUCCESS

Terminal:
```sh
Note: The following floating-point exceptions are signalling: IEEE_UNDERFLOW_FLAG IEEE_DENORMAL
```
`9md.out`
```out
 ------------------------------------------------------------------------------


      A V E R A G E S   O V E R    1000 S T E P S


 NSTEP =  1000000   TIME(PS) =    4000.000  TEMP(K) =   298.00  PRESS =     0.0
 Etot   =   -154306.3281  EKtot   =     26577.2718  EPtot      =   -180883.5999
 BOND   =       863.0466  ANGLE   =      2305.1215  DIHED      =      1579.9836
 UB     =         0.0000  IMP     =         0.0000  CMAP       =       517.6718
 1-4 NB =      1107.3648  1-4 EEL =     11339.1205  VDWAALS    =     20499.5637
 EELEC  =   -219095.4724  EHBOND  =         0.0000  RESTRAINT  =         0.0000
 EKCMT  =         0.0000  VIRIAL  =         0.0000  VOLUME     =    430147.3490
                                                    Density    =         1.0413
 ------------------------------------------------------------------------------


      R M S  F L U C T U A T I O N S


 NSTEP =  1000000   TIME(PS) =    4000.000  TEMP(K) =     1.41  PRESS =     0.0
 Etot   =       226.8889  EKtot   =       125.6572  EPtot      =       188.7256
 BOND   =        24.7281  ANGLE   =        35.7943  DIHED      =        20.5028
 UB     =         0.0000  IMP     =         0.0000  CMAP       =        15.9884
 1-4 NB =        13.8377  1-4 EEL =        41.6883  VDWAALS    =       206.1437
 EELEC  =       315.5087  EHBOND  =         0.0000  RESTRAINT  =         0.0000
 EKCMT  =         0.0000  VIRIAL  =         0.0000  VOLUME     =       851.3498
                                                    Density    =         0.0021
 ------------------------------------------------------------------------------

| MC Barostat:      10000 volume changes attempted.
| MC Barostat:       3342 changes successful ( 33.42%)
 ------------------------------------------------------------------------------

--------------------------------------------------------------------------------
   5.  TIMINGS
--------------------------------------------------------------------------------

|  NonSetup CPU Time in Major Routines:
|
|     Routine           Sec        %
|     ------------------------------
|     Nonbond          76.89    4.15
|     Bond              0.00    0.00
|     Angle             0.00    0.00
|     Dihedral          0.00    0.00
|     Shake             2.75    0.15
|     RunMD          1760.29   95.03
|     Other            12.39    0.67
|     ------------------------------
|     Total          1852.32

|  PME Nonbond Pairlist CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     Set Up Cit           0.00    0.00
|     Build List           0.00    0.00
|     ---------------------------------
|     Total                0.00    0.00

|  PME Direct Force CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     NonBonded Calc       0.00    0.00
|     Exclude Masked       0.00    0.00
|     Other                1.16    0.06
|     ---------------------------------
|     Total                1.16    0.06

|  PME Reciprocal Force CPU Time:
|
|     Routine              Sec        %
|     ---------------------------------
|     1D bspline           0.00    0.00
|     Grid Charges         0.00    0.00
|     Scalar Sum           0.00    0.00
|     Gradient Sum         0.00    0.00
|     FFT                  0.00    0.00
|     ---------------------------------
|     Total                0.00    0.00

|  Final Performance Info:
|     -----------------------------------------------------
|     Average timings for last   23000 steps:
|     Elapsed(s) =      42.03 Per Step(ms) =       1.83
|         ns/day =      47.28   seconds/ns =    1827.48
|
|     Average timings for all steps:
|     Elapsed(s) =    1853.86 Per Step(ms) =       1.85
|         ns/day =      46.61   seconds/ns =    1853.86
|     -----------------------------------------------------

|  Setup CPU time:            0.51 seconds
|  NonSetup CPU time:      1852.32 seconds
|  Total CPU time:         1852.83 seconds     0.51 hours

|  Setup wall time:           0    seconds
|  NonSetup wall time:     1854    seconds
|  Total wall time:        1854    seconds     0.52 hours
```

## Convert `parm7` and `rst7` to `pdb` for visualization

`9md_cpptraj_netCDF_to_pdb.out`
```out
	Reading 'amber_o98_ion.parm7' as Amber Topology
	Radius Set: modified Bondi radii (mbondi)
INPUT: Reading input from '9md_cpptraj_netCDF_to_pdb.in'
  [trajin 9md.rst7]
	Reading '9md.rst7' as Amber NC Restart
  [trajout 9md_rst7.pdb PDB]
	Writing '9md_rst7.pdb' as PDB
---------- RUN BEGIN -------------------------------------------------

PARAMETER FILES (1 total):
 0: amber_o98_ion.parm7, 56610 atoms, 13351 res, box: Truncated octahedron, 13059 mol, 12955 solvent

INPUT TRAJECTORIES (1 total):
 0: '9md.rst7' is a NetCDF AMBER restart file with coordinates, velocities, time, box, Parm amber_o98_ion.parm7 (Truncated octahedron box) (reading 1 of 1)
  Coordinate processing will occur on 1 frames.

OUTPUT TRAJECTORIES (1 total):
  '9md_rst7.pdb' (1 frames) is a PDB file

BEGIN TRAJECTORY PROCESSING:
Warning: No PDB space group specified.
Warning: Topology 'amber_o98_ion.parm7' has 12955 extra points
Warning:   that will not be included in output PDB.
Warning: To include them, specify 'include_ep'. Otherwise, use output PDB as
Warning:   topology or create a new topology with 'strip' or 'parmstrip'.
.....................................................
ACTIVE OUTPUT TRAJECTORIES (1):
  9md_rst7.pdb (coordinates, velocities, time, box)
----- 9md.rst7 (1-1, 1) -----
100% Complete.

Read 1 frames and processed 1 frames.
TIME: Avg. throughput= 6.8052 frames / second.

ACTION OUTPUT:
TIME: Analyses took 0.0000 seconds.

RUN TIMING:
TIME:		Init               : 0.0000 s (  0.00%)
TIME:		Trajectory Process : 0.1469 s ( 99.88%)
TIME:		Action Post        : 0.0000 s (  0.00%)
TIME:		Analysis           : 0.0000 s (  0.00%)
TIME:		Data File Write    : 0.0000 s (  0.00%)
TIME:		Other              : 0.0002 s (  0.00%)
TIME:	Run Total 0.1471 s
---------- RUN END ---------------------------------------------------
TIME: Total execution time: 0.2131 seconds.
--------------------------------------------------------------------------------
```

`9md_rst7.pdb` (Opaque, cyan) overlaid with `8md_rst7.pdb` (BrushedMetal, grey) on VMD\
![9md_8md_rst_pdb](B2/trial5_tleap_script_parm7_rst7/9md_8md_rst_pdb.png)\
Almost completely deviated?!

`9md_rst7.pdb` (Opaque, cyan) overlaid with `amber_o98_wions_water.pdb` (BrushedMetal, grey) on VMD\
![9md_o98_ion_rst_pdb](B2/trial5_tleap_script_parm7_rst7/9md_o98_ion_pdb.png)\
Complete "translation" (major deviations) of molecule compared to the input structure!

`9md_rst.pdb` with solvent box `QuickSurf`\
![9md_rst_pdb_quicksurf](B2/trial5_tleap_script_parm7_rst7/9md_rst_pdb_quicksurf.png)\
Solvent box still present but looks close to disintegrating

# USE `9md.rst7` AND `amber_o98_ion.parm7` IN NEXT STEP (`04_production_md.ipynb`)
Before 2024/06/17

## Trial 7

Match all equilibration parameters to `chen2023`

Universal changes (where applicable):
1. Non-bonded cutoff: `cut = 9.0`
2. Constant pressure: `barostat = 1` (Berendsen; might not be very good...)
3. Target (reference) temperature: `temp0 = 310.0` (310 K, 37 degrees C)

Fixed key settings descriptors

### 1. Minimization
Change maximum cycles to `maxcyc = 35000`

```in
minimization of solvent
 &cntrl
  imin = 1, maxcyc = 35000,   
  ncyc = 20, ntx = 1,                     
  ntwe = 0, ntwr = 500, ntpr = 50,
  ntb = 1, ntp = 0,
  cut = 9.0,   
  ntr=1, restraintmask = ':1-294',
  restraint_wt = 100.,
  ioutfm=1, ntxo=2,
 /

```
Key settings:
```
imin = 1       This run is a minimization run.
maxcyc = 35000  The maximum amount of minimization cycles is 35000.
ncyc = 20      The first 20 cycles will utilize the steepest descent 
                  algorithm before shifting to the conjugate gradient 
                  algorithm for the remaining cycles. 
ntx = 1        The coordinates but not velocities are read formatted 
                  from the coordinate file provided. 
ntwe = 0       No mden files are written.
ntwr = 500     The amount of steps in which "restrt" files are written.
ntpr = 50      The amount of steps in which "mdout" and "mdinfo" files
                  are written.
ntc = 2        Bonds involving H are constrained. 
ntf = 2        Bonds involving H are omitted from force evaluation. 
ntb = 1        There is constant volume.
ntp = 0        There is no pressure scaling.
cut = 9.0     The non-bonded cutoff is 9.0 Angstroms.
ntr = 1        Use restraints.
restraintmask = ':1-294'  Restrain the solute - protein (res 1-294).
restraint_wt = 100  Positional restraint is 100 kcal/mol*Ang^-2.
ioutfm = 1     The format of the coordinate and velocity trajectory 
                 files written as binary NetCDF.
ntxo=2         The "restrt" file format is NetCDF.
```

### 2. Heating
NOTE: 
1. Constant volume `ntb = 1`
2. Equilibrate for 75 ps (`nstlim - istep2 = 75000`) at `temp0`

```in
&cntrl
  imin = 0, nstlim = 1000000, dt = 0.001,
  irest = 0, ntx = 1, ig = -1,
  tempi = 100.0, temp0 = 310.0,
  ntc = 2, ntf = 2, tol = 0.00001,
  ntwx = 10000, ntwe = 0, ntwr = 1000, ntpr = 1000,
  cut = 9.0, iwrap = 0,
  ntt =3, gamma_ln=1.0, ntb = 1, ntp = 0,
  nscm = 0,
  ntr=1, restraintmask=':1-294', restraint_wt=100.0
  nmropt=1,
  ioutfm=1, ntxo=2,
 /
&wt TYPE="TEMP0", istep1=0, istep2=925000, value1=100.0, value2=310.0, 
/
&wt TYPE="END", 
/

```
Key settings:
```
imin = 0          This is not a minimization run.
nstlim = 1000000  There will be 1000000 MD-steps. 
dt = 0.001        There is a time step of 1 femtosecond. 
irest = 0         The simulation will not be restarted.
ntx = 1           The coordinates but not velocities are read 
                    formatted from the coordinate file provided.
ig = -1           The random seed number is based on the current 
                    date and time of the run.  
tempi = 100.0     The initial temperature is 100.0 K.
temp0 = 310.0     The reference temperature is set to 310.0 K 
ntc = 2           Bonds involving H are constrained. 
ntf = 2           Bonds involving H are omitted from force evaluation. 
tol = 0.00001     The error of tolerance is 0.00001 Angstroms.
ntwx = 10000      The coordinates are written to a mdcrd file 10000 times. 
ntwe = 0          No mden files are written.
ntwr = 1000       Number of steps in which "restrt" files are written. 
ntpr = 1000       Number of steps in which "mdout" and "mdinfo" 
                    files are written.
cut = 9.0         The non-bonded cutoff is 9.0 Angstroms.
iwrap = 0         No wrapping is performed.
ntt = 3           Langevin thermostat for temperature control is set.
gamma_ln=1.       The collision frequency gamma is set to 1 picosecond. 
ntb = 1           There is constant volume.
ntp = 0           There is no pressure scaling.
nscm = 0          There is no removal of center of mass motion.
ntr=1             Atoms are restrained.
restraintmask = ':1-81' Restrain solute (as in 1min.in).
restraint_wt = 100  Positional restraint is 100 kcal/mol*Ang^-2.
nmropt=1          NMR restraints and weight changes will be read. 
ioutfm = 1        The format of the coordinate and velocity trajectory
                    files written as binary NetCDF.
ntxo=2            The "restrt" file format is NetCDF.

TYPE="TEMP0"                  The target T will vary.
istep1=0, istep2=925000      Change in T will occur in 925000 increments.
value1=100.0, value2=310.0      T begins at 100.0 K and increases to 310.0 K.
```

`chen2023`: Equilibrate at 310 K under constant volume for 75 ps after temperature has finished increasing to 310 K\
$\rightarrow$ calculate how long the temperature should be increased

In [1]:
1000000 - 75 * 1000

925000

Temperature increment "halt" works as expected\
Some fluctuations, but NOT major
```out

 NMR restraints: Bond =    0.000   Angle =     0.000   Torsion =     0.000
===============================================================================

 NSTEP =   924000   TIME(PS) =     924.000  TEMP(K) =   308.15  PRESS =     0.0
 Etot   =   -226434.8166  EKtot   =     41360.2578  EPtot      =   -267795.0744
 BOND   =       704.4259  ANGLE   =      1650.3002  DIHED      =      1407.7933
 UB     =         0.0000  IMP     =         0.0000  CMAP       =       387.3439
 1-4 NB =      1571.6599  1-4 EEL =     12403.1352  VDWAALS    =     34319.2730
 EELEC  =   -322523.0659  EHBOND  =         0.0000  RESTRAINT  =      2284.0602
 EAMBER (non-restraint)  =   -270079.1346
 ------------------------------------------------------------------------------

 NMR restraints: Bond =    0.000   Angle =     0.000   Torsion =     0.000
===============================================================================

 NSTEP =   925000   TIME(PS) =     925.000  TEMP(K) =   308.58  PRESS =     0.0
 Etot   =   -226605.3607  EKtot   =     41418.5156  EPtot      =   -268023.8763
 BOND   =       683.0184  ANGLE   =      1652.9852  DIHED      =      1402.9721
 UB     =         0.0000  IMP     =         0.0000  CMAP       =       385.6168
 1-4 NB =      1561.9943  1-4 EEL =     12389.6891  VDWAALS    =     33975.6865
 EELEC  =   -322303.0928  EHBOND  =         0.0000  RESTRAINT  =      2227.2539
 EAMBER (non-restraint)  =   -270251.1302
 ------------------------------------------------------------------------------

 NMR restraints: Bond =    0.000   Angle =     0.000   Torsion =     0.000
===============================================================================

 NSTEP =   926000   TIME(PS) =     926.000  TEMP(K) =   306.70  PRESS =     0.0
 Etot   =   -226867.6235  EKtot   =     41166.4062  EPtot      =   -268034.0297
 BOND   =       705.8346  ANGLE   =      1683.3232  DIHED      =      1418.2692
 UB     =         0.0000  IMP     =         0.0000  CMAP       =       388.6755
 1-4 NB =      1570.4305  1-4 EEL =     12385.7500  VDWAALS    =     34379.3765
 EELEC  =   -322859.7271  EHBOND  =         0.0000  RESTRAINT  =      2294.0380
 EAMBER (non-restraint)  =   -270328.0677
 ------------------------------------------------------------------------------

 NMR restraints: Bond =    0.000   Angle =     0.000   Torsion =     0.000
===============================================================================

 NSTEP =   927000   TIME(PS) =     927.000  TEMP(K) =   307.28  PRESS =     0.0
 Etot   =   -226849.8370  EKtot   =     41243.5664  EPtot      =   -268093.4034
 BOND   =       663.3237  ANGLE   =      1631.9784  DIHED      =      1409.6638
 UB     =         0.0000  IMP     =         0.0000  CMAP       =       386.4089
 1-4 NB =      1576.3026  1-4 EEL =     12394.5510  VDWAALS    =     34289.9441
 EELEC  =   -322713.9045  EHBOND  =         0.0000  RESTRAINT  =      2268.3286
 EAMBER (non-restraint)  =   -270361.7320
 ------------------------------------------------------------------------------

 NMR restraints: Bond =    0.000   Angle =     0.000   Torsion =     0.000
===============================================================================

 NSTEP =   928000   TIME(PS) =     928.000  TEMP(K) =   310.04  PRESS =     0.0
 Etot   =   -226541.3814  EKtot   =     41615.1094  EPtot      =   -268156.4907
 BOND   =       666.4801  ANGLE   =      1666.4165  DIHED      =      1413.1823
 UB     =         0.0000  IMP     =         0.0000  CMAP       =       389.0512
 1-4 NB =      1578.0509  1-4 EEL =     12423.7373  VDWAALS    =     34342.0129
 EELEC  =   -322906.5546  EHBOND  =         0.0000  RESTRAINT  =      2271.1327
 EAMBER (non-restraint)  =   -270427.6234
 ------------------------------------------------------------------------------

 NMR restraints: Bond =    0.000   Angle =     0.000   Torsion =     0.000
===============================================================================

 NSTEP =   929000   TIME(PS) =     929.000  TEMP(K) =   311.35  PRESS =     0.0
 Etot   =   -226219.4142  EKtot   =     41789.6211  EPtot      =   -268009.0353
 BOND   =       654.5694  ANGLE   =      1648.1167  DIHED      =      1405.6675
 UB     =         0.0000  IMP     =         0.0000  CMAP       =       393.1738
 1-4 NB =      1562.0661  1-4 EEL =     12394.5819  VDWAALS    =     34401.3668
 EELEC  =   -322744.6460  EHBOND  =         0.0000  RESTRAINT  =      2276.0686
 EAMBER (non-restraint)  =   -270285.1039
 ------------------------------------------------------------------------------

 NMR restraints: Bond =    0.000   Angle =     0.000   Torsion =     0.000
===============================================================================

 NSTEP =   930000   TIME(PS) =     930.000  TEMP(K) =   309.24  PRESS =     0.0
 Etot   =   -226329.9939  EKtot   =     41506.5977  EPtot      =   -267836.5916
 BOND   =       694.0090  ANGLE   =      1667.5517  DIHED      =      1416.2329
 UB     =         0.0000  IMP     =         0.0000  CMAP       =       392.2898
 1-4 NB =      1551.6753  1-4 EEL =     12379.5640  VDWAALS    =     34020.0530
 EELEC  =   -322301.6156  EHBOND  =         0.0000  RESTRAINT  =      2343.6483
 EAMBER (non-restraint)  =   -270180.2399
 ------------------------------------------------------------------------------
```

### 3. Relax the system at a constant pressure

```in
&cntrl
  imin = 0, nstlim = 1000000, dt = 0.001,
  irest = 1, ntx = 5, ig = -1,
  temp0 = 310.0,
  ntc = 2, ntf = 2, tol = 0.00001,
  ntwx = 10000, ntwe = 0, ntwr = 1000, ntpr = 1000,
  cut = 9.0, iwrap = 0,
  ntt =3, gamma_ln=1.0, ntb = 2, ntp = 1,
  nscm = 0, barostat = 1,
  ntr=1, restraintmask=':1-294', restraint_wt=100.0
  ioutfm=1, ntxo=2,
 /

```
Key settings:
```
imin = 0          This run is not a minimization run.
nstlim = 1000000  There will be 1000000 MD-steps. 
dt = 0.001        There is a time step of 1 fs. 
irest = 1         Restart the simulation from previously 
                    saved restart files.          
ntx = 5           The coordinates and velocities are 
                    read into the run.
ig = -1           The random seed number is based on the 
                    current date and time of the run.
temp0 = 310.0     The reference temperature is set to 310 K.
ntwx = 10000      The coordinates are written to a mdcrd file 10000 times. 
cut = 9.0         The non-bonded cutoff is 9.0 Angstroms.
ntt = 3           Langevin thermostat for temperature control is set.
gamma_ln=1.       The collision frequency gamma is set to 1 picosecond. 
ntb = 2           Constant pressure.
ntp = 1           Use isotropic position scaling.
barostat = 1      Use the Berendsen barostat. 
nscm = 0          No removal of center of mass motion.
ntr=1             Atoms are restrained.
restraintmask = ':1-294' Restrain solute (as in previous steps).
restraint_wt = 100  Positional restraint is 100 kcal/mol*Ang^-2 (same as above).
nmropt=1          NMR restraints and weight changes will be read.
ioutfm = 1        The format of the coordinate and velocity trajectory
                    files written as binary NetCDF.
ntxo=2            The "restrt" file format is NetCDF.
```

### 4. Lower the restraints on the system

```in
&cntrl
  imin = 0, nstlim = 1000000, dt = 0.001,
  irest = 1, ntx = 5, ig = -1,
  temp0 = 310.0,
  ntc = 2, ntf = 2, tol = 0.00001,
  ntwx = 10000, ntwe = 0, ntwr = 1000, ntpr = 1000,
  cut = 9.0, iwrap = 0,
  ntt =3, gamma_ln=1.0, ntb = 2, ntp = 1,
  nscm = 0, barostat = 1,
  ntr=1, restraintmask=':1-294', restraint_wt=10.0
  ioutfm=1, ntxo=2,
 /

```

### 5. Minimize the system with restraints just on the backbone of the molecule

```in
Minimization of everything excluding backbone
 &cntrl
  imin = 1, maxcyc = 35000,
  ncyc = 30, ntx = 1, 
  ntwe = 0, ntwr = 500, ntpr = 50,
  ntc = 2, ntf = 2, ntb = 1, ntp = 0,
  cut = 9.0, 
  ntr=1, restraintmask="@CA,N,C", restraint_wt=10.0
  ioutfm=1, ntxo=2,
 /

```

### 6. Relax system at a constant pressure with backbone restraints

```in
&cntrl
  imin = 0, nstlim = 1000000, dt = 0.001,
  irest = 0, ntx = 1, ig = -1,
  tempi = 310.0, temp0 = 310.0,
  ntc = 2, ntf = 2, tol = 0.00001,
  ntwx = 10000, ntwe = 0, ntwr = 1000, ntpr = 1000,
  cut = 9.0, iwrap = 0,
  ntt =3, gamma_ln=1.0, ntb = 2, ntp = 1, 
  nscm = 0, barostat = 1,
  ntr=1, restraintmask="@CA,N,C", restraint_wt=10.0
  ioutfm=1, ntxo=2,
 /

```
Same `irest` and `ntx` settings as `2mdheat.in` $\rightarrow$ standard operation after minimization?

### 7. Reduce the backbone restraint force

```in
&cntrl
  imin = 0, nstlim = 1000000, dt = 0.001,
  irest = 1, ntx = 5, ig = -1,
  temp0 = 310.0,
  ntc = 2, ntf = 2, tol = 0.00001,
  ntwx = 10000, ntwe = 0, ntwr = 1000, ntpr = 1000,
  cut = 9.0, iwrap = 0,
  ntt =3, gamma_ln=1.0, ntb = 2, ntp = 1,
  nscm = 0, barostat = 1,
  ntr=1, restraintmask="@CA,N,C", restraint_wt=1.0
  ioutfm=1, ntxo=2,
 /

```

### 8. Continue to reduce the backbone restraint force

```in
&cntrl
  imin = 0, nstlim = 1000000, dt = 0.001,
  irest = 1, ntx = 5, ig = -1,
  temp0 = 310.0,
  ntc = 2, ntf = 2, tol = 0.00001,
  ntwx = 10000, ntwe = 0, ntwr = 1000, ntpr = 1000,
  cut = 9.0, iwrap = 0,
  ntt =3, gamma_ln=1.0, ntb = 2, ntp = 1,
  nscm = 0, barostat = 1,
  ntr=1, restraintmask="@CA,N,C", restraint_wt=0.1
  ioutfm=1, ntxo=2,
 /

```

### 9. Relax the system with no restraints

```in
&cntrl   
  imin = 0, nstlim = 1000000, dt = 0.001,
  irest = 1, ntx = 5, ig = -1,
  temp0 = 310.0,
  ntc = 2, ntf = 2, tol = 0.00001,
  ntwx = 10000, ntwe = 0, ntwr = 1000, ntpr = 1000,
  cut = 9.0, iwrap = 0,
  ntt =3, gamma_ln=1.0, ntb = 2, ntp = 1,
  nscm = 1000, barostat = 1,
  ioutfm=1, ntxo=2,
 /

```

### 10. Submit a job script to run all 9 steps

Cluster is taking forever to start `RUNNING` (predicted `PENDING` until `2024-06-20T17:46:51`, today 18/06/2024)\
$\rightarrow$ move back to local machine to run equilibration... (OK to equilibrate AMBER20 `tleap` protein system on AMBER22?)\
$\rightarrow$ cluster started ~3 hours late but fininshed running before local machine!

Terminal command
```sh
bash all_relax.sh &
```
`all_relax.sh`
```sh
export AMBERHOME=/home/t38guest/amber22
export CUDA_VISIBLE_DEVICES=0

cd /home/wjoon21/project/chen2023_possible_allosteric/B2/trial7_cubic_temp_barostat_slurm

echo "starting 1min at $(date)"

$AMBERHOME/bin/pmemd -O -i 1min.in\
            -o 1min.out -p amber_o98_cubic_ion.parm7 -c amber_o98_cubic_ion.rst7 -r 1min.rst7\
            -inf 1min.info -ref amber_o98_cubic_ion.rst7 -x mdcrd.1min
echo "ending 1min at $(date)"

echo "starting 2mdheat at $(date)"

$AMBERHOME/bin/pmemd.cuda -O -i 2mdheat.in\
            -o 2mdheat.out -p amber_o98_cubic_ion.parm7 -c 1min.rst7 -r 2mdheat.rst7\
            -inf 2mdheat.info -ref 1min.rst7 -x mdcrd.2mdheat
echo "ending 2mdheat at $(date)"

echo "starting 3md at $(date)"

$AMBERHOME/bin/pmemd.cuda -O -i 3md.in\
            -o 3md.out -p amber_o98_cubic_ion.parm7 -c 2mdheat.rst7 -r 3md.rst7\
            -inf 3md.info -ref 2mdheat.rst7 -x mdcrd.3md
echo "ending 3md at $(date)"

echo "starting 4md at $(date)"

$AMBERHOME/bin/pmemd.cuda -O -i 4md.in\
            -o 4md.out -p amber_o98_cubic_ion.parm7 -c 3md.rst7 -r 4md.rst7\
            -inf 4md.info -ref 3md.rst7 -x mdcrd.4md
echo "ending 4md at $(date)"

echo "starting 5min at $(date)"

$AMBERHOME/bin/pmemd -O -i 5min.in\
            -o 5min.out -p amber_o98_cubic_ion.parm7 -c 4md.rst7 -r 5min.rst7\
            -inf 5min.info -ref 4md.rst7 -x mdcrd.5min
echo "ending 5min at $(date)"

echo "starting 6md at $(date)"

$AMBERHOME/bin/pmemd.cuda -O -i 6md.in\
            -o 6md.out -p amber_o98_cubic_ion.parm7 -c 5min.rst7 -r 6md.rst7\
            -inf 6md.info -ref 5min.rst7 -x mdcrd.6md
echo "ending 6md at $(date)"

echo "starting 7md at $(date)"

$AMBERHOME/bin/pmemd.cuda -O -i 7md.in\
            -o 7md.out -p amber_o98_cubic_ion.parm7 -c 6md.rst7 -r 7md.rst7\
            -inf 7md.info -ref 6md.rst7 -x mdcrd.7md
echo "ending 7md at $(date)"

echo "starting 8md at $(date)"

$AMBERHOME/bin/pmemd.cuda -O -i 8md.in\
            -o 8md.out -p amber_o98_cubic_ion.parm7 -c 7md.rst7 -r 8md.rst7\
            -inf 8md.info -ref 7md.rst7 -x mdcrd.8md
echo "ending 8md at $(date)"

echo "starting 9md at $(date)"

$AMBERHOME/bin/pmemd.cuda -O -i 9md.in\
            -o 9md.out -p amber_o98_cubic_ion.parm7 -c 8md.rst7 -r 9md.rst7\
            -inf 9md.info -ref 8md.rst7 -x mdcrd.9md
echo "ending 9md at $(date)"
```

NOTE: 
1. `.mdcrd` files are only generated (even if they are specified in the `pmemd` command) when `imin=0` (i.e., NOT minimization)
2. `imin=1` (minimization) is only supported by `pmemd` (AMBER22 manual 22.6.1. Supported Features) (i.e., NOT `pmemd.cuda`)